** **
# **snRNAseq bioinformatics: Pre-processing, quality control and cell-type annotation in _Xenopus tropicalis_ late gastrula**

- https://scanpy.readthedocs.io/en/stable/index.html (documentation for single cell analysis in Python)
- Stage 13 single nucleus RNAseq data (Parse technology), unpublished

 






** **
# __Overview__

Section 1) Preprocessing, quality control and doublet detection

Section 2) Linear & non-linear dimension reduction techniques

Section 3) Surveying marker gene expression & doublet validation

Section 4) Clustering algorithms and gene expression based annotation

Section 5) Differential gene expression analysis

Section 6) Final cell-type annotation rationale considering on differential expression and literature precedence


** **

# __Section 1: Preprocessing & Quality Control__

 __Topics__
- Loading data and background software
- Filtering genes/cell
- Filtering cells with aberrant mitochondrial and ribosomal gene expression
- Doublet filtering
- Data normalization
- Optional batch-effect correction examples (commented)
- Calling highly variable genes
- Optional regression example (commented)

** **
 

__Part 1: Loading background software and data__

In [ ]:
#Load background software
import numpy as np
import pandas as pd
import scanpy as sc
import matplotlib.pyplot as plt
import matplotlib
#import mnnpy
import scrublet as scr
import seaborn as sns

In [ ]:
#Set scanpy settings for visualizations
sc.settings.verbosity = 2 #Scanpy verbosity reporting
sc.settings.set_figure_params(dpi=75) #Figure pixel resolution settings

# Suppress verbose package deprecation warnings during plotting

import warnings

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)

__Project-relative paths and output directories__


In [ ]:
# --- Stage- and condition-specific project path setup ---
from pathlib import Path
import sys

STAGE = "Stage13"
CONDITION = None
DATASET_SLUG = "Stage13-WT"

STAGE_FOLDER_NAMES = {
    "Stage10": "Stage10",
    "Stage11.25": "Stage11_25",
    "Stage12": "Stage12",
    "Stage12.0": "Stage12_0",
    "Stage12.5": "Stage12_5",
    "Stage13": "Stage13",
}

CONDITION_SPECIFIC_STAGES = {"Stage11.25", "Stage12.0"}
VALID_CONDITIONS = {"WT", "MO"}

if STAGE not in STAGE_FOLDER_NAMES:
    raise ValueError(
        f"Invalid STAGE: {STAGE}. Choose from: {list(STAGE_FOLDER_NAMES)}"
    )

STAGE_FOLDER = STAGE_FOLDER_NAMES[STAGE]

if STAGE in CONDITION_SPECIFIC_STAGES:
    if CONDITION is None:
        raise ValueError(f"{STAGE} requires CONDITION='WT' or CONDITION='MO'.")
    CONDITION = CONDITION.upper()
    if CONDITION not in VALID_CONDITIONS:
        raise ValueError(f"Invalid CONDITION: {CONDITION}")
else:
    CONDITION = None


def find_project_root(start=None):
    """Walk upward until the repository root is found."""
    start = Path.cwd() if start is None else Path(start).resolve()

    for path in [start] + list(start.parents):
        if (path / ".git").exists() or (path / "README.md").exists():
            return path

    raise FileNotFoundError(
        "Could not find the project root. Run this notebook from inside "
        "the cloned repository."
    )


PROJECT_DIR = find_project_root()

if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))

DATA_DIR = PROJECT_DIR / "data"
RAW_DATA_DIR = DATA_DIR / "raw"
EXAMPLE_DATA_DIR = DATA_DIR / "example"
METADATA_DIR = DATA_DIR / "metadata"

STAGE_RAW_DATA_DIR = RAW_DATA_DIR / STAGE_FOLDER
STAGE_METADATA_DIR = METADATA_DIR / STAGE_FOLDER

RESULTS_DIR = PROJECT_DIR / "results"
STAGE_RESULTS_BASE_DIR = RESULTS_DIR / STAGE_FOLDER

if CONDITION is not None:
    STAGE_RESULTS_DIR = STAGE_RESULTS_BASE_DIR / CONDITION
else:
    STAGE_RESULTS_DIR = STAGE_RESULTS_BASE_DIR

QC_DIR = STAGE_RESULTS_DIR / "qc"
FIGURES_DIR = STAGE_RESULTS_DIR / "figures"
PROCESSED_DATA_DIR = STAGE_RESULTS_DIR / "processed_data"
DEG_DIR = STAGE_RESULTS_DIR / "differential_expression"
VALIDATION_DIR = STAGE_RESULTS_DIR / "validation"
TABLES_DIR = STAGE_RESULTS_DIR / "tables"

SECTION_FIGURE_DIRS = {
    "Section1": FIGURES_DIR / "Section1-Preprocessing",
    "Section2": FIGURES_DIR / "Section2-DimensionReduction",
    "Section3": FIGURES_DIR / "Section3-MarkerGeneExpression",
    "Section4": FIGURES_DIR / "Section4-ClusteringAnnotation",
    "Section5": FIGURES_DIR / "Section5-DifferentialExpression",
    "Section6": FIGURES_DIR / "Section6-FinalAnnotation",
}

folders_to_create = [
    RAW_DATA_DIR,
    EXAMPLE_DATA_DIR,
    METADATA_DIR,
    STAGE_RAW_DATA_DIR,
    STAGE_METADATA_DIR,
    RESULTS_DIR,
    STAGE_RESULTS_BASE_DIR,
    STAGE_RESULTS_DIR,
    QC_DIR,
    FIGURES_DIR,
    PROCESSED_DATA_DIR,
    DEG_DIR,
    VALIDATION_DIR,
    TABLES_DIR,
    *SECTION_FIGURE_DIRS.values(),
]

for folder in folders_to_create:
    folder.mkdir(parents=True, exist_ok=True)

SECTION_FIGURES_DIR = SECTION_FIGURE_DIRS["Section1"]
sc.settings.figdir = str(SECTION_FIGURES_DIR)

print("Project directory:", PROJECT_DIR)
print("Dataset:", DATASET_SLUG)
print("Input directory:", STAGE_RAW_DATA_DIR)
print("Results directory:", STAGE_RESULTS_DIR)
print("Current figure directory:", SECTION_FIGURES_DIR)


__Upload our Stage 13 expression matrix and meta data__

In [ ]:
# Load the stage-specific AnnData file from a project-relative path.
INPUT_H5AD_FILENAME = "PARSE-WTSt13-Unfiltered_FromSeurat.h5ad"
INPUT_H5AD = STAGE_RAW_DATA_DIR / INPUT_H5AD_FILENAME

if not INPUT_H5AD.exists():
    raise FileNotFoundError(
        f"Input file not found:\n{INPUT_H5AD}\n\n"
        f"Place the input file in:\n{STAGE_RAW_DATA_DIR}"
    )

adata = sc.read_h5ad(INPUT_H5AD)

print("Dataset:", DATASET_SLUG)
print("Loaded file:", INPUT_H5AD)
print("AnnData shape:", adata.shape)

adata


In [ ]:
#Visualize features our dataframe
adata.obs

__Part 1: Filtering by cells/gene and genes/cell__

In [ ]:
# Filter background barcodes
sc.pp.filter_cells(adata, min_genes=1000) # Remove cells expressing fewer than 1,000 genes

#I personally don't remove lowly expressed genes until after I see clustering.
#There may be a cluster with 50 or fewer cells which I don't want to arbitrarily exclude.

#sc.pp.filter_genes(adata, min_cells=50) #Remove genes who are expressed in 50 or fewer cells

print(adata.n_obs, adata.n_vars) #How many cells and genes pass QC

In [ ]:
adata

** **
__Part 2: Investigating and filtering mitochondrial and ribosomal reads from cells__

Typically in single Cell RNAseq:
- High percentages (>20%) of Mitochondrial reads indicate stressed and dying cells
- Low percentages (<3.5%) of Ribosomal reads indicate transcriptionally silent cells (for typical scRNAseq)

__However, in single nuclei RNAseq we do not expect to recover appreciable amount of ribosomal or mitochondrial reads.__




__Denote Ribosomal and Mitochondrial genes within the dataset and calculate their abundance metrics__

In [ ]:
#adata.var_names_make_unique()

In [ ]:
# Specifically label mitochondrial genes within the matrix
adata.var['mt'] = adata.var_names.str.startswith(('COX', 'ND', "CYTB", "ATP"))


# Specifically label ribosomal genes within the matrix
adata.var['ribo'] = adata.var_names.str.startswith(("rps","rpl"))


#Calculate "mt" and "ribo" percentage of count metrics
sc.pp.calculate_qc_metrics(adata, qc_vars=['mt','ribo'], percent_top=None, log1p=True, inplace=True)


#Plot our "mt" and "ribo" percent metrics
sc.pl.violin(adata, ['n_genes_by_counts', 'total_counts', 'pct_counts_mt','pct_counts_ribo'],
             jitter=0.4, groupby = "sample", rotation= 45, save="_S01_01_violin_sample_n_genes_by_counts_total_counts_pct_counts_mt_pct_counts_ribo.png")

In [ ]:
adata.var

In [ ]:
#adata.var.index.name = "gene"
#print(adata.var.index.name)

In [ ]:
# Make sure gene-level total_counts exists
# calculate_qc_metrics should add adata.var["total_counts"]
top_genes = (
    adata.var[["total_counts"]]
    .sort_values("total_counts", ascending=False)
    .head(30)
    .reset_index()
    .rename(columns={"index": "gene"})
)

plt.figure(figsize=(8, 6))
sns.barplot(
    data=top_genes,
    y="gene",
    x="total_counts"
)

plt.xlabel("Total counts")
plt.ylabel("Gene")
plt.title("Top 30 Expressed Genes")
plt.tight_layout()
plt.savefig(SECTION_FIGURES_DIR / "_cell009_top_30_expressed_genes.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
top_mt_genes = (
    adata.var.loc[adata.var["mt"], ["total_counts"]]
    .sort_values("total_counts", ascending=False)
    .head(30)
    .reset_index()
    .rename(columns={"index": "gene"})
)

plt.figure(figsize=(8, 6))
sns.barplot(
    data=top_mt_genes,
    y="gene",
    x="total_counts"
)

plt.xlabel("Total counts")
plt.ylabel("Mitochondrial gene")
plt.title("Top 30 Expressed Mitochondrial Genes")
plt.tight_layout()
plt.savefig(SECTION_FIGURES_DIR / "_cell010_top_30_expressed_mitochondrial_genes.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
#Optional but recommended: make ribosomal labeling case-insensitive
adata.var["ribo"] = adata.var_names.str.lower().str.startswith(("rps", "rpl"))

# Make sure total_counts exists in adata.var
# This should already exist after sc.pp.calculate_qc_metrics()
if "total_counts" not in adata.var.columns:
    sc.pp.calculate_qc_metrics(
        adata,
        qc_vars=["mt", "ribo"],
        percent_top=None,
        log1p=True,
        inplace=True
    )

# Get top ribosomal genes
top_ribo_genes = (
    adata.var.loc[adata.var["ribo"], ["total_counts"]]
    .sort_values("total_counts", ascending=False)
    .head(30)
    .reset_index()
    .rename(columns={"index": "gene"})
)

# Plot
plt.figure(figsize=(8, 7))
sns.barplot(
    data=top_ribo_genes,
    y="gene",
    x="total_counts"
)

plt.xlabel("Total counts")
plt.ylabel("Ribosomal gene")
plt.title("Top 30 Expressed Ribosomal Genes")
plt.tight_layout()
plt.savefig(SECTION_FIGURES_DIR / "_cell011_top_30_expressed_ribosomal_genes.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
plt.figure(figsize=(7, 5))
sns.histplot(
    adata.obs["pct_counts_ribo"],
    bins=50
)

plt.xlabel("Percent ribosomal counts")
plt.ylabel("Number of cells")
plt.title("Distribution of Ribosomal Count Percentage")
plt.tight_layout()
plt.savefig(SECTION_FIGURES_DIR / "_cell012_ribosomal_percentage_distribution.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
plt.figure(figsize=(7, 5))
sns.histplot(
    adata.obs["pct_counts_mt"],
    bins=50
)

plt.xlabel("Percent mitochondrial counts")
plt.ylabel("Number of cells")
plt.title("Distribution of mitochondrial Count Percentage")
plt.tight_layout()
plt.savefig(SECTION_FIGURES_DIR / "_cell013_mitochondrial_percentage_distribution.png", dpi=300, bbox_inches="tight")
plt.show()

__Remove cells with aberrant mitochondrial and ribosomal count proportions__

In [ ]:
#Remove cells with >20% mitochondrial read count percentage
adata = adata[adata.obs['pct_counts_mt'] < 20, :]

# Remove cells with >3.5% ribosomal read count percentage
adata = adata[adata.obs['pct_counts_ribo'] < 3.5, :]

print("Remaining cells %d"%adata.n_obs)

** **
__Part 3: Predicting and filtering Doublets__

In [ ]:
adata.raw = adata
scrub = scr.Scrublet(adata.raw.X)
adata.obs['doublet_scores'], adata.obs['predicted_doublets'] = scrub.scrub_doublets()
scrub.plot_histogram()

sum(adata.obs['predicted_doublets'])

In [ ]:
# add in column with singlet/doublet instead of True/False
adata.obs['doublet_info'] = adata.obs["predicted_doublets"].astype(str)

__Visualize the estimated doublet proportion of the dataset__

In [ ]:
#Plot doublets vs singlets to reveal relative doublet abundance
sc.pl.violin(adata, 'n_genes_by_counts',
             jitter=0.4, groupby = 'doublet_info', rotation=45, save="_S01_02_violin_doublet_info_n_genes_by_counts.png")

__So the question is... to remove, or not to remove doublets from the dataset?__

__In my opinion, before you remove your doublets it's best to double check the doublet prediction by visualizing known marker gene expression!__

Rationalization: If a cell expresses multiple markers from distinct tissue types it's probably a doublet.

** **
__Part 4: Additional pre-processing, calling Highly Variable Genes and correcting for batch effects__

__Normalization__

In [ ]:
# normalize to depth of 10,000 counts/cell
sc.pp.normalize_per_cell(adata, counts_per_cell_after=1e4)

# log-transform
sc.pp.log1p(adata)


# store normalized counts in the raw slot, 
# we will subset adata.X for variable genes, but want to keep all genes matrix as well.
adata.raw = adata

adata

__Distinguish Highly Variable Genes__

In [ ]:
# compute variable genes
sc.pp.highly_variable_genes(adata, min_mean=0.005, max_mean=5, min_disp=0.005)
print("Highly variable genes: %d"%sum(adata.var.highly_variable))

#plot variable genes
sc.pl.highly_variable_genes(adata, save="_S01_03_highly_variable_genes.png")

# subset for variable genes in the dataset
adata = adata[:, adata.var['highly_variable']]

__Conduct library Batch Corrections using MNN software (FYI)__

We only have a single sequencing library for Stage 10.5, however if you did have multiple sequencing libraries then you could perform batch corrections.

In the example code below, the batch key called "lib_name" is a column of adata2.obs where the row identifiers were "Library_17", "Library_18", "Library_19", etc

For you to run this code using your data, you need to change the column "lib_name" to your library column name, and the "Library_17"-"Library_22" identifiers to your row identifiers.

In [ ]:
#adata2 = adata.raw.to_adata() 
#var_genes_all = adata.var.highly_variable
#print("Highly variable genes: %d"%sum(var_genes_all))

#Detect HVG in each different library via batch_key parameter
#sc.pp.highly_variable_genes(adata2, min_mean=0.0125, max_mean=3, min_disp=0.5, batch_key = 'lib_name')

#print("Highly variable genes intersection: %d"%sum(adata2.var.highly_variable_intersection))
#print("Number of batches where gene is variable:")
#print(adata2.var.highly_variable_nbatches.value_counts())
#var_genes_batch = adata2.var.highly_variable_nbatches > 0
#Compare overlapping variably expressed genes across each of the libraries
#Here you rename the HVG into another variable name, take note if you want to make figures from them
#print("Any batch var genes: %d"%sum(var_genes_batch))
#print("All data var genes: %d"%sum(var_genes_all))
#print("Overlap: %d"%sum(var_genes_batch & var_genes_all))
#print("Variable genes in all batches: %d"%sum(adata2.var.highly_variable_nbatches == 6))
#print("Overlap batch intersection and all: %d"%sum(var_genes_all & adata2.var.highly_variable_intersection))
#Select and subset genes which remain variable across at least 2/3 libraries
#var_select = adata2.var.highly_variable_nbatches > 2
#var_genes = var_select.index[var_select]
#len(var_genes)
#Create an individual AnnData object from each library
# split per batch into new objects.
#batches = adata.obs['lib_name'].cat.categories.tolist()
#alldata = {}
#for batch in batches:
    #alldata[batch] = adata2[adata2.obs['lib_name'] == batch,]
#alldata

#Finally, perform batch correction via mnnpy software
#cdata = sc.external.pp.mnn_correct(alldata['Library_17'], alldata['Library_18'], alldata['Library_19'],
                                  #alldata['Library_20'], alldata['Library_21'], alldata['Library_22'],
                                  #svd_dim = 50, batch_key = 'lib_name', save_raw = True, var_subset = var_genes)

#Corr_data is the new combined AnnData object's subsetted Variable Genes
#corr_data = cdata[0][:,var_genes]
#corr_data.X.shape


__Utilize regression to eliminate impact of mitochondrial counts__

I'm not implementing it here because we have so few mitochondrial counts, however FYI

In [ ]:
# Utilize linear regression to decrease effect of differential or unwanted variables
#sc.pp.regress_out(adata, ['pct_counts_mt'])


** **
** **

# __Section 2: Linear and non-linear dimension reduction analysis__

 __Topics__
- Principal Component Analysis (PCA)
- t-Stochastic Neighbor Embedding (tSNE)
- Uniform Manifold Approximation Projection (UMAP)

** **
 

In [ ]:
# Route all Scanpy and Matplotlib figures in Section 2 here.
SECTION_FIGURES_DIR = SECTION_FIGURE_DIRS["Section2"]
SECTION_FIGURES_DIR.mkdir(parents=True, exist_ok=True)
sc.settings.figdir = str(SECTION_FIGURES_DIR)

print("Current section figure directory:", SECTION_FIGURES_DIR)


__Part 1: Linear Dimension Reduction: Principal Component Analysis (PCA)__

__Calculate PCA__

In [ ]:
# Perform PCA on our pre-processed data

sc.tl.pca(adata, svd_solver='arpack', n_comps=100)

__Visualize top PCA components__

- During PCA, components are ranked in descending order of variance explained.
- Here, covariance is derived from comparing a cell's differential expression compared to other cells across the dataset.
- Generally, genes that are the most highly expressed and differentially expressed will contribute the most variance.

In [ ]:
# Investigate specific principal components

sc.pl.pca(adata, color='sample', components = ['1,2','3,4','5,6','7,8'], ncols=2, save="_S02_01_pca_sample.png")
#components comparisons can be changed to your liking

__Visualize the variance contributed by each Principal Component__

In [ ]:
#Investigate the variance explained by each Principle Coordinate
sc.pl.pca_variance_ratio(adata, log=True, n_pcs = 40, save="_S02_02_pca_variance_ratio.png")

__Visualize the PCA loadings__

- Principle component loadings reveal the genes which contribute most to variance of each component

In [ ]:
#Plot loadings
sc.pl.pca_loadings(adata, components=[1,2,3,4,5,6,7,8], save="_S02_03_pca_loadings.png")

** **
__Part 2: Non-linear Dimension Reduction: t-Stochastic Neighbor Embedding (tSNE)__

__Compute tSNE__

In [ ]:
sc.tl.tsne(adata, n_pcs = 30)

#This can be run using different n_pcs which varies the number of principal components used in the analysis

__Visualize tSNE__

In [ ]:
sc.pl.tsne(adata, save="_S02_04_tsne.png")

__Try varying the number of components used to educate tSNE, and see how it effects the embeddings__

In [ ]:
#Change the n_pcs = (X) parameter and see the effect
sc.tl.tsne(adata, n_pcs = 99) #Here I have changed to 8 principal components

In [ ]:
#Plot your varied components tSNE
sc.pl.tsne(adata, save="_S02_05_tsne.png")
#This can be run using different n_pcs which varies the number of principal components used in the analysis

** **
__Part 3: Non-linear Dimension Reduction: Uniform Manifold Approximation Projection (UMAP)__

__Calculate UMAP__

You can modulate your UMAP output by selecting different parameters of principal coordinate (n_pcs) and nearest neighbors (n_neighbors)

- Increasing n_pcs incorporates more components that explain less variance.
- Increasing n_neighbors causes each cell to consider additional neighbors when placing itself in the plot.

In [ ]:
#Calculate a neighborhood graph based on principal components and some number of nearest neighbors to consider
sc.pp.neighbors(adata, n_pcs = 10, n_neighbors = 50)

#Compute machine learning using UMAP algorithm combined with our data set and our neighborhood graph calculated from the previous step
sc.tl.umap(adata, n_components=2)

#Plot the UMAP
sc.pl.umap(adata, save="_S02_06_umap.png")

In [ ]:
sc.pp.neighbors(adata, n_pcs = 70, n_neighbors = 25)
sc.tl.umap(adata, n_components=2)
sc.pl.umap(adata, frameon=False, size=10, add_outline=True, save="_S02_07_umap.png")

In [ ]:
sc.pp.neighbors(adata, n_pcs = 99, n_neighbors = 50)
sc.tl.umap(adata, n_components=2)
sc.pl.umap(adata, frameon=False, size=10, add_outline=True, save="_S02_08_umap.png")

In [ ]:
# we can also plot the umap with neighbor edges
sc.pl.umap(adata, title="UMAP", edges=True, save="_S02_09_umap.png")

** **

# __Section 3: Surveying marker gene expression & doublet validation__

 __Topics__
- Visualizing the localization of known germ layer differentiating marker genes
- Marker gene expression helps us validate doublet predictions

** **
 

In [ ]:
# Route all Scanpy and Matplotlib figures in Section 3 here.
SECTION_FIGURES_DIR = SECTION_FIGURE_DIRS["Section3"]
SECTION_FIGURES_DIR.mkdir(parents=True, exist_ok=True)
sc.settings.figdir = str(SECTION_FIGURES_DIR)

print("Current section figure directory:", SECTION_FIGURES_DIR)


__Part 1: Visualizing germ layer marker genes__

In [ ]:
#Epidermal Markers Markers
gene_list=['krt7', 'grhl1']
sc.pl.umap(adata, color=gene_list,  color_map=plt.cm.turbo, add_outline=True, size=12, frameon=False, legend_fontsize=12, alpha=1, save="_S03_01_umap_marker_panel.png")

In [ ]:
#Neuroectoderm Markers
gene_list=['sox2', 'sox3', 'pax3']
sc.pl.umap(adata, color=gene_list,  color_map=plt.cm.turbo, add_outline=True, size=12, frameon=False, legend_fontsize=12, alpha=1, save="_S03_02_umap_marker_panel.png")

In [ ]:
#Mesodermal Markers Markers
gene_list=['tbxt', 'fgf8']
sc.pl.umap(adata, color=gene_list,  color_map=plt.cm.turbo, add_outline=True, size=12, frameon=False, legend_fontsize=12, alpha=1, save="_S03_03_umap_marker_panel.png")

In [ ]:
#Endodermal Markers Markers
gene_list=['darmin', 'gata6']
sc.pl.umap(adata, color=gene_list,  color_map=plt.cm.turbo, add_outline=True, size=12, frameon=False, legend_fontsize=12, alpha=1, save="_S03_04_umap_marker_panel.png")

__Part 2: Validating doublet predictions__

In [ ]:
#Predicted Doublets
sc.pl.umap(adata, color=["doublet_info", 'total_counts'],  color_map=plt.cm.turbo, add_outline=True, size=12, frameon=False, legend_fontsize=12, alpha=1, save="_S03_05_umap_doublet_info_total_counts.png")

__Removing predicted doublets__

In [ ]:
# Remove doublets from our data
adata = adata.raw.to_adata()  # also revert back to the raw counts as the main matrix in adata

adata = adata[adata.obs['doublet_info'] == 'False',:]
print(adata.shape)

In [ ]:
#Predicted Doublets
sc.pl.umap(adata, color="doublet_info",  color_map=plt.cm.turbo, add_outline=True, size=12, frameon=False, legend_fontsize=12, alpha=1, save="_S03_06_umap_doublet_info.png")

In [ ]:
sc.pl.umap(adata, color=['COX1'])

In [ ]:
sc.tl.leiden(adata, resolution = 0.3, key_added = "leiden_0.3")

In [ ]:
sc.pl.umap(adata, color=['leiden_0.3'])

In [ ]:
# Remove cluster 0
adata = adata[adata.obs['leiden_0.3'].astype(str) != '12'].copy()

# Remove the unused category label
adata.obs['leiden_0.3'] = adata.obs['leiden_0.3'].cat.remove_unused_categories()

# Plot remaining clusters
sc.pl.umap(adata, color='leiden_0.3')

__Re-running UMAP clustering without doublet influence__

In [ ]:
#sc.pp.neighbors(adata, n_pcs=20, n_neighbors=60)
#sc.tl.umap(adata, n_components=2)
#sc.pl.umap(adata, frameon=False, size=10, add_outline=True, save="_S03_07_umap.png")

** **

# __Section 4: Clustering algorithms and marker gene expression based annotation__

 __Topics__
- Leiden clustering algorithm
- Subsetting clusters & annotating cells based on Marker Gene threshold Expression

** **
 

In [ ]:
# Route all Scanpy and Matplotlib figures in Section 4 here.
SECTION_FIGURES_DIR = SECTION_FIGURE_DIRS["Section4"]
SECTION_FIGURES_DIR.mkdir(parents=True, exist_ok=True)
sc.settings.figdir = str(SECTION_FIGURES_DIR)

print("Current section figure directory:", SECTION_FIGURES_DIR)


__Part 1: Utilize the Leiden algorithm for broadly annotating cell clusters__

__Choose your Leiden resolution and run the algorithm on our adata's manifold projection (UMAP)__

In [ ]:
sc.tl.leiden(adata, key_added = "leiden_1.0") # default resolution in 1.0
sc.tl.leiden(adata, resolution = 0.05, key_added = "leiden_0.05")
sc.tl.leiden(adata, resolution = 0.1, key_added = "leiden_0.1")
sc.tl.leiden(adata, resolution = 0.2, key_added = "leiden_0.2")
sc.tl.leiden(adata, resolution = 0.8, key_added = "leiden_0.8")
sc.tl.leiden(adata, resolution = 1.2, key_added = "leiden_1.2")
sc.tl.leiden(adata, resolution = 10.0, key_added = "leiden_10.0")

__Visualize the different Leiden resolutions__

In [ ]:
sc.pl.umap(adata, color=['leiden_0.05', 'leiden_0.1', 'leiden_0.2',  'leiden_0.8', 'leiden_1.2'], save="_S04_01_umap_leiden_0_05_leiden_0_1_leiden_0_2_leiden_0_8.png")

__Part 2: Subsetting and Expression Based Annotation of leiden_0.2 - cluster "1", "7", "8", "10" and "11"__

__First we subset cluster "1" , "7", "8", "10" and "11" from leiden_0.2__

In [ ]:
adata_1 = adata[
    adata.obs["leiden_0.2"].astype(str).isin(["1", "7", "8", "10", "11"])
].copy()



In [ ]:
sc.pl.umap(adata_1, frameon=False, size=10, add_outline=True, save="_S04_02_umap.png")

__Confirming Epidermal, Neuroectodermal, Mesodermal and Endodermal Marker expression__

In [ ]:
#Markers Expression Check
gene_list=['krt7', 'grhl1', 'sox2', 'sox3', 'tbxt', 'fgf8', 'foxj1', 'darmin', 'gata6', 'slc5a8', 'ventx1.2', 'agr2', 'ag1', 'pax3', 'bicc1', 'cirop', 'itln1', 'itln2']
sc.pl.umap(adata_1, color=gene_list,  color_map=plt.cm.turbo, add_outline=True, size=12, frameon=False, legend_fontsize=12, alpha=1, save="_S04_03_umap_marker_panel.png")

__Annotating cell's based on a threshold of marker gene expression__

In [ ]:
# Define a threshold for "high" expression
threshold = 1  # adjust based on your data

# Cells where both agr2 and pitx1 are above the threshold
mask = (
    (adata_1[:, "agr2"].X > threshold).toarray().flatten()
    & (adata_1[:, "pitx1"].X > threshold).toarray().flatten()
)

# Reset all cells in adata_1 to Other
adata_1.obs["region_annotation"] = "Other"

# Assign Cement Gland to double-positive cells
adata_1.obs.loc[
    mask,
    "region_annotation"
] = "Cement Gland Primordium"

# Plot
sc.pl.umap(
    adata_1,
    color="region_annotation",
    add_outline=True,
    palette="tab20"
)

In [ ]:
import pandas as pd
import scanpy as sc
import numpy as np
from scipy import sparse

# Define expression threshold for itln1
threshold_itln1 = 0.5
annotation_col = "region_annotation"
new_label = "Goblet Cell"

# Extract itln1 expression
itln1_expression = adata_1[:, "itln1"].X

if sparse.issparse(itln1_expression):
    itln1_expression = itln1_expression.toarray()

itln1_mask = (
    np.asarray(itln1_expression).ravel()
    > threshold_itln1
)

# Ensure the annotation column is categorical
if not isinstance(
    adata_1.obs[annotation_col].dtype,
    pd.CategoricalDtype
):
    adata_1.obs[annotation_col] = pd.Categorical(
        adata_1.obs[annotation_col]
    )

# Add Goblet Cell as an allowed category if necessary
if new_label not in adata_1.obs[annotation_col].cat.categories:
    adata_1.obs[annotation_col] = (
        adata_1.obs[annotation_col]
        .cat.add_categories([new_label])
    )

# Label every itln1-positive cell as Goblet Cell
adata_1.obs.loc[
    itln1_mask,
    annotation_col
] = new_label

# Check counts
print("Goblet Cell assigned:", itln1_mask.sum())
print(adata_1.obs[annotation_col].value_counts())

# Visualize the result
sc.pl.umap(
    adata_1,
    color=annotation_col,
    add_outline=True,
    palette="tab20"
)

In [ ]:
import numpy as np
import pandas as pd
import scanpy as sc
from scipy import sparse

# Define expression threshold for krt70
threshold_krt70 = 1.0
annotation_col = "region_annotation"
new_label = "Epidermis 1"

# Extract krt70 expression from adata_1
krt70_expression = adata_1[:, "krt70"].X

if sparse.issparse(krt70_expression):
    krt70_expression = krt70_expression.toarray()

krt70_mask = (
    np.asarray(krt70_expression).ravel()
    > threshold_krt70
)

# Ensure the annotation column is categorical
if not isinstance(
    adata_1.obs[annotation_col].dtype,
    pd.CategoricalDtype
):
    adata_1.obs[annotation_col] = pd.Categorical(
        adata_1.obs[annotation_col]
    )

# Add Outer NNE as an allowed category if necessary
if new_label not in adata_1.obs[annotation_col].cat.categories:
    adata_1.obs[annotation_col] = (
        adata_1.obs[annotation_col]
        .cat.add_categories([new_label])
    )

# Protect Goblet Cell and Cement Gland annotations
not_goblet_or_cement = (
    ~adata_1.obs[annotation_col]
    .isin(["Goblet Cell", "Cement Gland Primordium"])
    .to_numpy()
)

# High krt70 expression and not protected
nne_mask = krt70_mask & not_goblet_or_cement

# Assign Outer NNE
adata_1.obs.loc[
    nne_mask,
    annotation_col
] = new_label

# Check counts
print("Epidermis 1 assigned:", nne_mask.sum())
print(adata_1.obs[annotation_col].value_counts())

# Visualize
sc.pl.umap(
    adata_1,
    color=annotation_col,
    add_outline=True,
    palette="tab20"
)

In [ ]:
import numpy as np
import pandas as pd
import scanpy as sc
from scipy import sparse

# Define expression thresholds
threshold_foxj1 = 0.5
threshold_tp73 = 0.5

annotation_col = "region_annotation"
new_label = "Ciliated Epidermal Progenitor"

# Extract foxj1 expression
foxj1_expression = adata_1[:, "foxj1"].X

if sparse.issparse(foxj1_expression):
    foxj1_expression = foxj1_expression.toarray()

foxj1_mask = (
    np.asarray(foxj1_expression).ravel()
    > threshold_foxj1
)

# Extract tp73 expression
tp73_expression = adata_1[:, "tp73"].X

if sparse.issparse(tp73_expression):
    tp73_expression = tp73_expression.toarray()

tp73_mask = (
    np.asarray(tp73_expression).ravel()
    > threshold_tp73
)

# Require both foxj1 and tp73 expression
ciliated_mask = foxj1_mask & tp73_mask

# Ensure the annotation column is categorical
if not isinstance(
    adata_1.obs[annotation_col].dtype,
    pd.CategoricalDtype
):
    adata_1.obs[annotation_col] = pd.Categorical(
        adata_1.obs[annotation_col]
    )

# Add the new category if necessary
if new_label not in adata_1.obs[annotation_col].cat.categories:
    adata_1.obs[annotation_col] = (
        adata_1.obs[annotation_col]
        .cat.add_categories([new_label])
    )

# Assign the new label and overwrite any existing annotation
adata_1.obs.loc[
    ciliated_mask,
    annotation_col
] = new_label

# Check counts
print(
    "Ciliated Epidermal Progenitor assigned:",
    ciliated_mask.sum()
)
print(adata_1.obs[annotation_col].value_counts())

# Visualize updated annotations
sc.pl.umap(
    adata_1,
    color=annotation_col,
    add_outline=True,
    palette="tab20"
)

In [ ]:
import numpy as np
import pandas as pd
import scanpy as sc
from scipy import sparse

# Define expression threshold for sox2
threshold_sox2 = 1.5

annotation_col = "region_annotation"
new_label = "Neural Plate 1"

# Extract sox2 expression from adata_1
sox2_expression = adata_1[:, "sox2"].X

if sparse.issparse(sox2_expression):
    sox2_expression = sox2_expression.toarray()

sox2_mask = (
    np.asarray(sox2_expression).ravel()
    > threshold_sox2
)

# Ensure the annotation column is categorical
if not isinstance(
    adata_1.obs[annotation_col].dtype,
    pd.CategoricalDtype
):
    adata_1.obs[annotation_col] = pd.Categorical(
        adata_1.obs[annotation_col]
    )

# Add Outer Neural Plate as an allowed category
if new_label not in adata_1.obs[annotation_col].cat.categories:
    adata_1.obs[annotation_col] = (
        adata_1.obs[annotation_col]
        .cat.add_categories([new_label])
    )

# Only allow Other and Outer NNE cells to be overwritten
allowed_labels_mask = (
    adata_1.obs[annotation_col]
    .isin(["Other", "Epidermis 1"])
    .to_numpy()
)

# High sox2 expression and an eligible existing annotation
outer_neural_plate_mask = (
    sox2_mask
    & allowed_labels_mask
)

# Assign Outer Neural Plate
adata_1.obs.loc[
    outer_neural_plate_mask,
    annotation_col
] = new_label

# Check counts
print(
    "Neural Plate 1 assigned:",
    outer_neural_plate_mask.sum()
)
print(adata_1.obs[annotation_col].value_counts())

# Visualize
sc.pl.umap(
    adata_1,
    color=annotation_col,
    add_outline=True,
    palette="tab20"
)

In [ ]:
import numpy as np
import pandas as pd
import scanpy as sc
from scipy import sparse

# Define expression threshold for pax3
threshold_pax3 = 1.0

annotation_col = "region_annotation"
new_label = "Neural Plate Border 1"

# Extract pax3 expression from adata_1
pax3_expression = adata_1[:, "pax3"].X

if sparse.issparse(pax3_expression):
    pax3_expression = pax3_expression.toarray()

pax3_mask = (
    np.asarray(pax3_expression).ravel()
    > threshold_pax3
)

# Ensure the annotation column is categorical
if not isinstance(
    adata_1.obs[annotation_col].dtype,
    pd.CategoricalDtype
):
    adata_1.obs[annotation_col] = pd.Categorical(
        adata_1.obs[annotation_col]
    )

# Add Neural Plate Border 1 as an allowed category
if new_label not in adata_1.obs[annotation_col].cat.categories:
    adata_1.obs[annotation_col] = (
        adata_1.obs[annotation_col]
        .cat.add_categories([new_label])
    )

# Only allow Other and Epidermis 1 cells to be overwritten
allowed_labels_mask = (
    adata_1.obs[annotation_col]
    .isin(["Other", "Epidermis 1"])
    .to_numpy()
)

# High pax3 expression and an eligible existing annotation
neural_plate_border_mask = (
    pax3_mask
    & allowed_labels_mask
)

# Assign Neural Plate Border 1
adata_1.obs.loc[
    neural_plate_border_mask,
    annotation_col
] = new_label

# Check counts
print(
    "Neural Plate Border 1 assigned:",
    neural_plate_border_mask.sum()
)
print(adata_1.obs[annotation_col].value_counts())

# Visualize
sc.pl.umap(
    adata_1,
    color=annotation_col,
    add_outline=True,
    palette="tab20"
)

In [ ]:
import numpy as np
import pandas as pd
import scanpy as sc
from scipy import sparse

# Define expression threshold for slc5a8
threshold_slc5a8 = 2.0

annotation_col = "region_annotation"
new_label = "Foregut Primordium"
protected_label = "Hindgut Primordium"

# Extract slc5a8 expression from adata_1
slc5a8_expression = adata_1[:, "slc5a8"].X

if sparse.issparse(slc5a8_expression):
    slc5a8_expression = slc5a8_expression.toarray()

slc5a8_mask = (
    np.asarray(slc5a8_expression).ravel()
    > threshold_slc5a8
)

# Ensure the annotation column is categorical
if not isinstance(
    adata_1.obs[annotation_col].dtype,
    pd.CategoricalDtype
):
    adata_1.obs[annotation_col] = pd.Categorical(
        adata_1.obs[annotation_col]
    )

# Add Foregut Primordium as an allowed category
if new_label not in adata_1.obs[annotation_col].cat.categories:
    adata_1.obs[annotation_col] = (
        adata_1.obs[annotation_col]
        .cat.add_categories([new_label])
    )

# Protect cells already labeled Hindgut Primordium
not_hindgut_mask = (
    adata_1.obs[annotation_col]
    .ne(protected_label)
    .to_numpy()
)

# High slc5a8 expression and not Hindgut Primordium
foregut_mask = slc5a8_mask & not_hindgut_mask

# Assign Foregut Primordium
adata_1.obs.loc[
    foregut_mask,
    annotation_col
] = new_label

# Check counts
print("Foregut Primordium assigned:", foregut_mask.sum())
print(
    "slc5a8-positive Hindgut cells protected:",
    (slc5a8_mask & ~not_hindgut_mask).sum()
)
print(adata_1.obs[annotation_col].value_counts())

# Visualize
sc.pl.umap(
    adata_1,
    color=annotation_col,
    add_outline=True,
    palette="tab20"
)

In [ ]:
import numpy as np
import pandas as pd
import scanpy as sc
from scipy import sparse

# Define expression threshold for ndrg1
threshold_ndrg1 = 1.0

annotation_col = "region_annotation"
new_label = "Midgut Primordium"

# Extract ndrg1 expression from adata_1
ndrg1_expression = adata_1[:, "ndrg1"].X

if sparse.issparse(ndrg1_expression):
    ndrg1_expression = ndrg1_expression.toarray()

# Cells expressing ndrg1 above the threshold
ndrg1_mask = (
    np.asarray(ndrg1_expression).ravel()
    > threshold_ndrg1
)

# Ensure the annotation column is categorical
if not isinstance(
    adata_1.obs[annotation_col].dtype,
    pd.CategoricalDtype
):
    adata_1.obs[annotation_col] = pd.Categorical(
        adata_1.obs[annotation_col]
    )

# Add Midgut Primordium as an allowed category
if new_label not in adata_1.obs[annotation_col].cat.categories:
    adata_1.obs[annotation_col] = (
        adata_1.obs[annotation_col]
        .cat.add_categories([new_label])
    )

# Assign Midgut Primordium regardless of the previous annotation
adata_1.obs.loc[
    ndrg1_mask,
    annotation_col
] = new_label

# Check counts
print("Midgut Primordium assigned:", ndrg1_mask.sum())
print(adata_1.obs[annotation_col].value_counts())

# Visualize
sc.pl.umap(
    adata_1,
    color=annotation_col,
    add_outline=True,
    palette="tab20"
)

In [ ]:
import numpy as np
import pandas as pd
import scanpy as sc
from scipy import sparse

# Define expression threshold for gatm
threshold_gatm = 2.0

annotation_col = "region_annotation"
new_label = "Hindgut Primordium"

# Extract gatm expression from adata_1
gatm_expression = adata_1[:, "gatm"].X

if sparse.issparse(gatm_expression):
    gatm_expression = gatm_expression.toarray()

# Cells expressing gatm above the threshold
gatm_mask = (
    np.asarray(gatm_expression).ravel()
    > threshold_gatm
)

# Ensure the annotation column is categorical
if not isinstance(
    adata_1.obs[annotation_col].dtype,
    pd.CategoricalDtype
):
    adata_1.obs[annotation_col] = pd.Categorical(
        adata_1.obs[annotation_col]
    )

# Add Hindgut Primordium as an allowed category
if new_label not in adata_1.obs[annotation_col].cat.categories:
    adata_1.obs[annotation_col] = (
        adata_1.obs[annotation_col]
        .cat.add_categories([new_label])
    )

# Allow overwriting only these existing annotations
allowed_to_overwrite = (
    adata_1.obs[annotation_col]
    .isin([
        "Foregut Primordium",
        "Other",
        "Epidermis 1"
    ])
    .to_numpy()
)

# High gatm expression and an eligible current annotation
hindgut_mask = gatm_mask & allowed_to_overwrite

# Assign Hindgut Primordium
adata_1.obs.loc[
    hindgut_mask,
    annotation_col
] = new_label

# Check counts
print("Hindgut Primordium assigned:", hindgut_mask.sum())
print(adata_1.obs[annotation_col].value_counts())

# Visualize
sc.pl.umap(
    adata_1,
    color=annotation_col,
    add_outline=True,
    palette="tab20"
)

In [ ]:
import numpy as np
import pandas as pd
import scanpy as sc
from scipy import sparse

# Define expression threshold for cirop
threshold_cirop = 1.0

annotation_col = "region_annotation"
new_label = "L/R Organizer Primordium"

# Extract cirop expression from adata_1
cirop_expression = adata_1[:, "cirop"].X

if sparse.issparse(cirop_expression):
    cirop_expression = cirop_expression.toarray()

# Cells expressing cirop above the threshold
cirop_mask = (
    np.asarray(cirop_expression).ravel()
    > threshold_cirop
)

# Ensure the annotation column is categorical
if not isinstance(
    adata_1.obs[annotation_col].dtype,
    pd.CategoricalDtype
):
    adata_1.obs[annotation_col] = pd.Categorical(
        adata_1.obs[annotation_col]
    )

# Add L/R Org. Primordium as an allowed category
if new_label not in adata_1.obs[annotation_col].cat.categories:
    adata_1.obs[annotation_col] = (
        adata_1.obs[annotation_col]
        .cat.add_categories([new_label])
    )

# Assign the label regardless of the current annotation
adata_1.obs.loc[
    cirop_mask,
    annotation_col
] = new_label

# Check counts
print("L/R Organizer Primordium assigned:", cirop_mask.sum())
print(adata_1.obs[annotation_col].value_counts())

# Visualize
sc.pl.umap(
    adata_1,
    color=annotation_col,
    add_outline=True,
    palette="tab20"
)

In [ ]:
import numpy as np
import pandas as pd
import scanpy as sc
from scipy import sparse

# Define expression threshold for hhex
threshold_hhex = 1.0

annotation_col = "region_annotation"
new_label = "Liver Primordium"

# Extract hhex expression from adata_1
hhex_expression = adata_1[:, "hhex"].X

if sparse.issparse(hhex_expression):
    hhex_expression = hhex_expression.toarray()

# Cells expressing hhex above the threshold
hhex_mask = (
    np.asarray(hhex_expression).ravel()
    > threshold_hhex
)

# Ensure the annotation column is categorical
if not isinstance(
    adata_1.obs[annotation_col].dtype,
    pd.CategoricalDtype
):
    adata_1.obs[annotation_col] = pd.Categorical(
        adata_1.obs[annotation_col]
    )

# Add Liver Primordium as an allowed category
if new_label not in adata_1.obs[annotation_col].cat.categories:
    adata_1.obs[annotation_col] = (
        adata_1.obs[annotation_col]
        .cat.add_categories([new_label])
    )

# Assign Liver Primordium regardless of the current annotation
adata_1.obs.loc[
    hhex_mask,
    annotation_col
] = new_label

# Check counts
print("Liver Primordium assigned:", hhex_mask.sum())
print(adata_1.obs[annotation_col].value_counts())

# Visualize
sc.pl.umap(
    adata_1,
    color=annotation_col,
    add_outline=True,
    palette="tab20"
)

In [ ]:
import numpy as np
import pandas as pd
import scanpy as sc
from scipy import sparse

# Define expression threshold for gsc
threshold_gsc = 1.0

annotation_col = "region_annotation"
new_label = "Anterior Pharyngeal Primordium"

# Extract gsc expression from adata_1
gsc_expression = adata_1[:, "gsc"].X

if sparse.issparse(gsc_expression):
    gsc_expression = gsc_expression.toarray()

# Cells expressing gsc above the threshold
gsc_mask = (
    np.asarray(gsc_expression).ravel()
    > threshold_gsc
)

# Ensure region_annotation is categorical
if not isinstance(
    adata_1.obs[annotation_col].dtype,
    pd.CategoricalDtype
):
    adata_1.obs[annotation_col] = pd.Categorical(
        adata_1.obs[annotation_col]
    )

# Add the new category if needed
if new_label not in adata_1.obs[annotation_col].cat.categories:
    adata_1.obs[annotation_col] = (
        adata_1.obs[annotation_col]
        .cat.add_categories([new_label])
    )

# Assign Anterior Pharyngeal Primordium regardless of current annotation
adata_1.obs.loc[
    gsc_mask,
    annotation_col
] = new_label

# Check counts
print(
    "Anterior Pharyngeal Primordium assigned:",
    gsc_mask.sum()
)
print(adata_1.obs[annotation_col].value_counts())

# Visualize updated annotations
sc.pl.umap(
    adata_1,
    color=annotation_col,
    add_outline=True,
    palette="tab20"
)

In [ ]:
import numpy as np
import pandas as pd
import scanpy as sc
from scipy import sparse

# Define expression threshold for nkx2-6
threshold_nkx26 = 1.0

annotation_col = "region_annotation"
new_label = "Pharyngeal Primordium"
gene = "nkx2-6"

# Extract nkx2-6 expression from adata_1
nkx26_expression = adata_1[:, gene].X

if sparse.issparse(nkx26_expression):
    nkx26_expression = nkx26_expression.toarray()

# Cells expressing nkx2-6 above the threshold
nkx26_mask = (
    np.asarray(nkx26_expression).ravel()
    > threshold_nkx26
)

# Ensure region_annotation is categorical
if not isinstance(
    adata_1.obs[annotation_col].dtype,
    pd.CategoricalDtype
):
    adata_1.obs[annotation_col] = pd.Categorical(
        adata_1.obs[annotation_col]
    )

# Add Pharyngeal Primordium as a category if needed
if new_label not in adata_1.obs[annotation_col].cat.categories:
    adata_1.obs[annotation_col] = (
        adata_1.obs[annotation_col]
        .cat.add_categories([new_label])
    )

# Assign Pharyngeal Primordium regardless of the current annotation
adata_1.obs.loc[
    nkx26_mask,
    annotation_col
] = new_label

# Check counts
print("Pharyngeal Primordium assigned:", nkx26_mask.sum())
print(adata_1.obs[annotation_col].value_counts())

# Visualize updated annotations
sc.pl.umap(
    adata_1,
    color=annotation_col,
    add_outline=True,
    palette="tab20"
)

In [ ]:
import numpy as np
import pandas as pd
import scanpy as sc
from scipy import sparse

# Define expression threshold for pdx1
threshold_pdx1 = 0.5

annotation_col = "region_annotation"
new_label = "Pancreatic Primordium"
gene = "pdx1"

# Extract pdx1 expression from adata_1
pdx1_expression = adata_1[:, gene].X

if sparse.issparse(pdx1_expression):
    pdx1_expression = pdx1_expression.toarray()

# Cells expressing pdx1 above the threshold
pdx1_mask = (
    np.asarray(pdx1_expression).ravel()
    > threshold_pdx1
)

# Ensure region_annotation is categorical
if not isinstance(
    adata_1.obs[annotation_col].dtype,
    pd.CategoricalDtype
):
    adata_1.obs[annotation_col] = pd.Categorical(
        adata_1.obs[annotation_col]
    )

# Add Pancreatic Primordium as a category if needed
if new_label not in adata_1.obs[annotation_col].cat.categories:
    adata_1.obs[annotation_col] = (
        adata_1.obs[annotation_col]
        .cat.add_categories([new_label])
    )

# Assign Pancreatic Primordium regardless of the current annotation
adata_1.obs.loc[
    pdx1_mask,
    annotation_col
] = new_label

# Check counts
print("Pancreatic Primordium assigned:", pdx1_mask.sum())
print(adata_1.obs[annotation_col].value_counts())

# Visualize updated annotations
sc.pl.umap(
    adata_1,
    color=annotation_col,
    add_outline=True,
    palette="tab20"
)

In [ ]:
import numpy as np
import pandas as pd
import scanpy as sc
from scipy import sparse

# Define expression threshold for foxi1
threshold_foxi1 = 1.0

annotation_col = "region_annotation"
new_label = "Ionocyte"
gene = "foxi1"

# Extract foxi1 expression from adata_1
foxi1_expression = adata_1[:, gene].X

if sparse.issparse(foxi1_expression):
    foxi1_expression = foxi1_expression.toarray()

# Cells expressing foxi1 above the threshold
foxi1_mask = (
    np.asarray(foxi1_expression).ravel()
    > threshold_foxi1
)

# Ensure region_annotation is categorical
if not isinstance(
    adata_1.obs[annotation_col].dtype,
    pd.CategoricalDtype
):
    adata_1.obs[annotation_col] = pd.Categorical(
        adata_1.obs[annotation_col]
    )

# Add Ionocyte as a category if needed
if new_label not in adata_1.obs[annotation_col].cat.categories:
    adata_1.obs[annotation_col] = (
        adata_1.obs[annotation_col]
        .cat.add_categories([new_label])
    )

# Assign Ionocyte regardless of the current annotation
adata_1.obs.loc[
    foxi1_mask,
    annotation_col
] = new_label

# Check counts
print("Ionocyte assigned:", foxi1_mask.sum())
print(adata_1.obs[annotation_col].value_counts())

# Visualize updated annotations
sc.pl.umap(
    adata_1,
    color=annotation_col,
    add_outline=True,
    palette="tab20"
)

__Performing an additional Knn calculation on annotation labels to eliminate the "Other" annotation__

In [ ]:
import scanpy as sc
import numpy as np
import pandas as pd

annotation_col = "region_annotation"
refined_annotation_col = "refined_region_annotation"

# Preserve the existing UMAP coordinates
original_umap = adata_1.obsm["X_umap"].copy()

# 1. Calculate a KNN graph using PCA coordinates
# This changes the neighbor graph, but does not recalculate UMAP positions
sc.pp.neighbors(
    adata_1,
    n_neighbors=10,
    use_rep="X_pca"
)

# 2. Save the original annotation labels
original_labels = (
    adata_1.obs[annotation_col]
    .astype(str)
    .copy()
)

# 3. Initialize the refined labels with the original labels
new_labels = original_labels.copy()

# 4. Access the KNN connectivity graph
connectivities = adata_1.obsp["connectivities"].tocsr()

# 5. Assign each cell the most common annotation among its neighbors
for cell_pos in range(adata_1.n_obs):

    neighbor_indices = (
        connectivities[cell_pos]
        .indices
    )

    # Exclude the cell itself if it appears in the neighbor graph
    neighbor_indices = neighbor_indices[
        neighbor_indices != cell_pos
    ]

    if len(neighbor_indices) == 0:
        continue

    neighbor_labels = original_labels.iloc[
        neighbor_indices
    ]

    most_common_labels = neighbor_labels.mode()

    if not most_common_labels.empty:
        new_labels.iloc[cell_pos] = (
            most_common_labels.iloc[0]
        )

# 6. Store the smoothed labels as a categorical column
adata_1.obs[refined_annotation_col] = pd.Categorical(
    new_labels
)

# Confirm that the UMAP coordinates were not changed
assert np.array_equal(
    adata_1.obsm["X_umap"],
    original_umap
)

# Check refined annotation counts
print(
    adata_1.obs[
        refined_annotation_col
    ].value_counts()
)

# 7. Visualize the refined labels on the unchanged UMAP
sc.pl.umap(
    adata_1,
    color=refined_annotation_col,
    add_outline=True,
    palette="tab20"
)

__Recombine our annotated subsetted data with our original adata UMAP under the column "region_annotation"__

In [ ]:
import pandas as pd

annotation_col = "region_annotation"

# Initialize the column if it does not exist
if annotation_col not in adata.obs.columns:
    adata.obs[annotation_col] = "Unknown"

# Convert to strings so new labels can be assigned freely
adata.obs[annotation_col] = adata.obs[annotation_col].astype(str)

# Find cells shared between adata and adata_1
shared_cells = adata.obs_names.intersection(adata_1.obs_names)

# Transfer annotations using aligned cell IDs
adata.obs.loc[shared_cells, annotation_col] = (
    adata_1.obs.loc[shared_cells, annotation_col]
    .astype(str)
    .values
)

# Convert back to categorical after all labels are transferred
adata.obs[annotation_col] = pd.Categorical(
    adata.obs[annotation_col]
)

# Check the results
print("Shared cells updated:", len(shared_cells))
print(adata.obs[annotation_col].value_counts())

In [ ]:
sc.pl.umap(adata, color='region_annotation', frameon=False, size=20, add_outline=False, alpha=.5, save="_S04_04_umap_region_annotation.png")

In [ ]:
sc.pl.umap(adata, color='leiden_0.2', frameon=False, size=20, add_outline=False, alpha=.5, save="_S04_05_umap_leiden_0_2.png")

__Part 3: Subsetting and Expression Based Annotation of leiden_0.2 - cluster "2", "3", "4" and "6"__

__Subset cluster 0 from leiden_0.2__

In [ ]:
adata_0 = adata[
    adata.obs["leiden_0.2"].astype(str).isin(['2', '3', '4', '6'])
].copy()



In [ ]:
sc.pl.umap(adata_0, frameon=False, size=20, add_outline=True, save="_S04_06_umap.png")

__Confirming Epidermal, Neuroectodermal, Mesodermal and Endodermal Marker expression__

In [ ]:
#Markers Expression Check
gene_list=['krt7', 'grhl1', 'sox2', 'sox3', 'tbxt', 'fgf8', 'darmin', 'gata6', 'klf5', 'pax3', 'tp63', 'six1', 'eya1', 'atp6v1b1', 'foxi1']
sc.pl.umap(adata_0, color=gene_list,  color_map=plt.cm.turbo, add_outline=True, size=20, frameon=False, legend_fontsize=12, alpha=1, save="_S04_07_umap_marker_panel.png")

__Annotating cell's based on a threshold of marker gene expression__

In [ ]:
# Define a threshold for high expression
threshold = 1.0

# Cells where both rax and pax6 exceed the threshold
mask = (
    (adata_0[:, "rax"].X > threshold).toarray().flatten()
    & (adata_0[:, "pax6"].X > threshold).toarray().flatten()
)

# Reset all cells in adata_0 to Other
adata_0.obs["region_annotation"] = "Other"

# Assign Eye Primordium to double-positive cells
adata_0.obs.loc[
    mask,
    "region_annotation"
] = "Eye Primordium"

# Check counts
print("Eye Primordium assigned:", mask.sum())
print(adata_0.obs["region_annotation"].value_counts())

# Plot
sc.pl.umap(
    adata_0,
    color="region_annotation",
    add_outline=True,
    palette="tab20"
)

In [ ]:
# Define expression threshold
threshold = 1.0
annotation_col = "region_annotation"
new_label = "Neural Plate 2"

# Extract sox2 expression
sox2_expression = adata_0[:, "sox2"].X

if sparse.issparse(sox2_expression):
    sox2_expression = sox2_expression.toarray()

sox2_mask = (
    np.asarray(sox2_expression).ravel()
    > threshold
)

# Extract sox3 expression
sox3_expression = adata_0[:, "sox3"].X

if sparse.issparse(sox3_expression):
    sox3_expression = sox3_expression.toarray()

sox3_mask = (
    np.asarray(sox3_expression).ravel()
    > threshold
)

# Require either sox2 or sox3 expression
sox_mask = sox2_mask | sox3_mask

# Ensure the annotation column is categorical
if not isinstance(
    adata_0.obs[annotation_col].dtype,
    pd.CategoricalDtype
):
    adata_0.obs[annotation_col] = pd.Categorical(
        adata_0.obs[annotation_col]
    )

# Add Neural Plate as an allowed category
if new_label not in adata_0.obs[annotation_col].cat.categories:
    adata_0.obs[annotation_col] = (
        adata_0.obs[annotation_col]
        .cat.add_categories([new_label])
    )

# Only overwrite cells currently labeled Other
other_mask = (
    adata_0.obs[annotation_col]
    .eq("Other")
    .to_numpy()
)

mask_to_update = sox_mask & other_mask

# Assign Neural Plate
adata_0.obs.loc[
    mask_to_update,
    annotation_col
] = new_label

# Check counts
print("Neural Plate assigned:", mask_to_update.sum())
print(adata_0.obs[annotation_col].value_counts())

# Visualize
sc.pl.umap(
    adata_0,
    color=annotation_col,
    add_outline=True,
    palette="tab20"
)

In [ ]:
# Define expression threshold for pax3
threshold = 1.0

annotation_col = "region_annotation"
new_label = "Neural Plate Border 2"

# Extract pax3 expression from adata_0
pax3_expression = adata_0[:, "pax3"].X

if sparse.issparse(pax3_expression):
    pax3_expression = pax3_expression.toarray()

pax3_mask = (
    np.asarray(pax3_expression).ravel()
    > threshold
)

# Ensure the annotation column is categorical
if not isinstance(
    adata_0.obs[annotation_col].dtype,
    pd.CategoricalDtype
):
    adata_0.obs[annotation_col] = pd.Categorical(
        adata_0.obs[annotation_col]
    )

# Add Neural Plate Border as an allowed category
if new_label not in adata_0.obs[annotation_col].cat.categories:
    adata_0.obs[annotation_col] = (
        adata_0.obs[annotation_col]
        .cat.add_categories([new_label])
    )

# Protect Neural Plate and Eye Primordium annotations
not_protected_mask = (
    ~adata_0.obs[annotation_col]
    .isin([
        "Neural Plate 2",
        "Eye Primordium"
    ])
    .to_numpy()
)

# High pax3 expression and not already assigned to a protected label
mask_to_update = pax3_mask & not_protected_mask

# Assign Neural Plate Border
adata_0.obs.loc[
    mask_to_update,
    annotation_col
] = new_label

# Check counts
print(
    "Neural Plate Border assigned:",
    mask_to_update.sum()
)
print(adata_0.obs[annotation_col].value_counts())

# Visualize
sc.pl.umap(
    adata_0,
    color=annotation_col,
    add_outline=True,
    palette="tab20"
)

In [ ]:
import numpy as np
import pandas as pd
import scanpy as sc
from scipy import sparse

# Define expression threshold
threshold = 0.5

annotation_col = "region_annotation"
new_label = "Neural Crest"

# Extract sox9 expression from adata_0
sox9_expression = adata_0[:, "sox9"].X

if sparse.issparse(sox9_expression):
    sox9_expression = sox9_expression.toarray()

# Cells expressing sox9 above the threshold
sox9_mask = (
    np.asarray(sox9_expression).ravel()
    > threshold
)

# Ensure the annotation column is categorical
if not isinstance(
    adata_0.obs[annotation_col].dtype,
    pd.CategoricalDtype
):
    adata_0.obs[annotation_col] = pd.Categorical(
        adata_0.obs[annotation_col]
    )

# Add Neural Crest as an allowed category
if new_label not in adata_0.obs[annotation_col].cat.categories:
    adata_0.obs[annotation_col] = (
        adata_0.obs[annotation_col]
        .cat.add_categories([new_label])
    )

# Protect Neural Plate and Eye Primordium
not_protected_mask = (
    ~adata_0.obs[annotation_col]
    .isin([
        "Neural Plate 2",
        "Eye Primordium"
    ])
    .to_numpy()
)

# High sox9 expression and not a protected annotation
neural_crest_mask = (
    sox9_mask
    & not_protected_mask
)

# Assign Neural Crest
adata_0.obs.loc[
    neural_crest_mask,
    annotation_col
] = new_label

# Check counts
print("Neural Crest assigned:", neural_crest_mask.sum())
print(adata_0.obs[annotation_col].value_counts())

# Visualize
sc.pl.umap(
    adata_0,
    color=annotation_col,
    add_outline=True,
    palette="tab20"
)

In [ ]:
import numpy as np
import pandas as pd
import scanpy as sc
import matplotlib.pyplot as plt
from scipy import sparse

# Define expression threshold for eya1
threshold_eya1 = 0.5

annotation_col = "region_annotation"
new_label = "Pre-placodal Region"

# Extract eya1 expression from adata_0
eya1_expression = adata_0[:, "eya1"].X

if sparse.issparse(eya1_expression):
    eya1_expression = eya1_expression.toarray()

# Cells expressing eya1 above the threshold
eya1_mask = (
    np.asarray(eya1_expression).ravel()
    > threshold_eya1
)

# Ensure the annotation column is categorical
if not isinstance(
    adata_0.obs[annotation_col].dtype,
    pd.CategoricalDtype
):
    adata_0.obs[annotation_col] = pd.Categorical(
        adata_0.obs[annotation_col]
    )

# Add six1/eya1 PPR as an allowed category
if new_label not in adata_0.obs[annotation_col].cat.categories:
    adata_0.obs[annotation_col] = (
        adata_0.obs[annotation_col]
        .cat.add_categories([new_label])
    )

# Protect Neural Plate Border and Neural Crest
excluded_labels = [
    "Neural Plate Border 2",
    "Neural Crest"
]

not_excluded_mask = (
    ~adata_0.obs[annotation_col]
    .isin(excluded_labels)
    .to_numpy()
)

# High eya1 expression and not a protected annotation
ppr_mask = eya1_mask & not_excluded_mask

# Assign six1/eya1 PPR
adata_0.obs.loc[
    ppr_mask,
    annotation_col
] = new_label

# Check counts
print("Pre-placodal Region assigned:", ppr_mask.sum())
print(adata_0.obs[annotation_col].value_counts())

# Visualize
sc.pl.umap(
    adata_0,
    color=annotation_col,
    add_outline=True,
    palette="tab20",
    color_map=plt.cm.turbo
)

In [ ]:
import numpy as np
import pandas as pd
import scanpy as sc
from scipy import sparse

# Define expression threshold for pax8
threshold_pax8 = 0.5

annotation_col = "region_annotation"
new_label = "Posterior Pre-placodal Region"

# Extract pax8 expression from adata_0
pax8_expression = adata_0[:, "pax8"].X

if sparse.issparse(pax8_expression):
    pax8_expression = pax8_expression.toarray()

# Cells expressing pax8 above the threshold
pax8_mask = (
    np.asarray(pax8_expression).ravel()
    > threshold_pax8
)

# Ensure the annotation column is categorical
if not isinstance(
    adata_0.obs[annotation_col].dtype,
    pd.CategoricalDtype
):
    adata_0.obs[annotation_col] = pd.Categorical(
        adata_0.obs[annotation_col]
    )

# Add pax8 PPR as an allowed category
if new_label not in adata_0.obs[annotation_col].cat.categories:
    adata_0.obs[annotation_col] = (
        adata_0.obs[annotation_col]
        .cat.add_categories([new_label])
    )

# Assign pax8 PPR regardless of the current annotation
adata_0.obs.loc[
    pax8_mask,
    annotation_col
] = new_label

# Check counts
print("Posterior Pre-placodal Region assigned:", pax8_mask.sum())
print(adata_0.obs[annotation_col].value_counts())

# Visualize
sc.pl.umap(
    adata_0,
    color=annotation_col,
    add_outline=True,
    palette="tab20"
)

In [ ]:
import numpy as np
import pandas as pd
import scanpy as sc
from scipy import sparse

# Define expression threshold for pitx1
threshold_pitx1 = 0.5

annotation_col = "region_annotation"
new_label = "Anterior Pre-placodal Region"

# Extract pitx1 expression from adata_0
pitx1_expression = adata_0[:, "pitx1"].X

if sparse.issparse(pitx1_expression):
    pitx1_expression = pitx1_expression.toarray()

# Cells expressing pitx1 above the threshold
pitx1_mask = (
    np.asarray(pitx1_expression).ravel()
    > threshold_pitx1
)

# Ensure the annotation column is categorical
if not isinstance(
    adata_0.obs[annotation_col].dtype,
    pd.CategoricalDtype
):
    adata_0.obs[annotation_col] = pd.Categorical(
        adata_0.obs[annotation_col]
    )

# Add pitx1 PPR as an allowed category
if new_label not in adata_0.obs[annotation_col].cat.categories:
    adata_0.obs[annotation_col] = (
        adata_0.obs[annotation_col]
        .cat.add_categories([new_label])
    )

# Assign pitx1 PPR regardless of the current annotation
adata_0.obs.loc[
    pitx1_mask,
    annotation_col
] = new_label

# Check counts
print("Anterior Pre-placodal Region assigned:", pitx1_mask.sum())
print(adata_0.obs[annotation_col].value_counts())

# Visualize
sc.pl.umap(
    adata_0,
    color=annotation_col,
    add_outline=True,
    palette="tab20"
)

In [ ]:
import numpy as np
import pandas as pd
import scanpy as sc
from scipy import sparse

# Define expression threshold for emx1l
threshold_emx1l = 0.5

annotation_col = "region_annotation"
new_label = "Anterior Neural Ridge"

# Extract emx1l expression from adata_0
emx1l_expression = adata_0[:, "emx1l"].X

if sparse.issparse(emx1l_expression):
    emx1l_expression = emx1l_expression.toarray()

# Cells expressing emx1l above the threshold
anr_mask = (
    np.asarray(emx1l_expression).ravel()
    > threshold_emx1l
)

# Ensure the annotation column is categorical
if not isinstance(
    adata_0.obs[annotation_col].dtype,
    pd.CategoricalDtype
):
    adata_0.obs[annotation_col] = pd.Categorical(
        adata_0.obs[annotation_col]
    )

# Add Anterior Neural Ridge as an allowed category
if new_label not in adata_0.obs[annotation_col].cat.categories:
    adata_0.obs[annotation_col] = (
        adata_0.obs[annotation_col]
        .cat.add_categories([new_label])
    )

# Assign Anterior Neural Ridge regardless of the current annotation
adata_0.obs.loc[
    anr_mask,
    annotation_col
] = new_label

# Check counts
print("Anterior Neural Ridge assigned:", anr_mask.sum())
print(adata_0.obs[annotation_col].value_counts())

# Visualize
sc.pl.umap(
    adata_0,
    color=annotation_col,
    add_outline=True,
    palette="tab20"
)

In [ ]:
import numpy as np
import pandas as pd
import scanpy as sc
from scipy import sparse

# Define expression threshold for tp63
threshold_tp63 = 0.5

annotation_col = "region_annotation"
new_label = "Epidermis 2"

# Extract tp63 expression from adata_0
tp63_expression = adata_0[:, "tp63"].X

if sparse.issparse(tp63_expression):
    tp63_expression = tp63_expression.toarray()

tp63_mask = (
    np.asarray(tp63_expression).ravel()
    > threshold_tp63
)

# Ensure the annotation column is categorical
if not isinstance(
    adata_0.obs[annotation_col].dtype,
    pd.CategoricalDtype
):
    adata_0.obs[annotation_col] = pd.Categorical(
        adata_0.obs[annotation_col]
    )

# Add Inner NNE Prog. as an allowed category
if new_label not in adata_0.obs[annotation_col].cat.categories:
    adata_0.obs[annotation_col] = (
        adata_0.obs[annotation_col]
        .cat.add_categories([new_label])
    )

# Only overwrite cells currently labeled Other
other_mask = (
    adata_0.obs[annotation_col]
    .eq("Other")
    .to_numpy()
)

mask_to_update = tp63_mask & other_mask

# Assign Inner NNE Prog.
adata_0.obs.loc[
    mask_to_update,
    annotation_col
] = new_label

# Check counts
print("Epidermis 2 assigned:", mask_to_update.sum())
print(adata_0.obs[annotation_col].value_counts())

# Visualize
sc.pl.umap(
    adata_0,
    color=annotation_col,
    add_outline=True,
    palette="tab20"
)

In [ ]:
import numpy as np
import pandas as pd
import scanpy as sc
from scipy import sparse

# Define expression threshold
threshold = 0.5

annotation_col = "region_annotation"
new_label = "Notoplate"

# Extract shh expression
shh_expression = adata_0[:, "shh"].X

if sparse.issparse(shh_expression):
    shh_expression = shh_expression.toarray()

shh_mask = (
    np.asarray(shh_expression).ravel()
    > threshold
)

# Extract ptch2 expression
ptch2_expression = adata_0[:, "ptch2"].X

if sparse.issparse(ptch2_expression):
    ptch2_expression = ptch2_expression.toarray()

ptch2_mask = (
    np.asarray(ptch2_expression).ravel()
    > threshold
)

# Require both shh and ptch2 expression
expression_mask = shh_mask & ptch2_mask

# Ensure the annotation column is categorical
if not isinstance(
    adata_0.obs[annotation_col].dtype,
    pd.CategoricalDtype
):
    adata_0.obs[annotation_col] = pd.Categorical(
        adata_0.obs[annotation_col]
    )

# Add Notoplate as an allowed category
if new_label not in adata_0.obs[annotation_col].cat.categories:
    adata_0.obs[annotation_col] = (
        adata_0.obs[annotation_col]
        .cat.add_categories([new_label])
    )

# Assign Notoplate regardless of the current annotation
adata_0.obs.loc[
    expression_mask,
    annotation_col
] = new_label

# Check counts
print("Notoplate assigned:", expression_mask.sum())
print(adata_0.obs[annotation_col].value_counts())

# Visualize
sc.pl.umap(
    adata_0,
    color=annotation_col,
    add_outline=True,
    palette="tab20"
)

In [ ]:
import numpy as np
import pandas as pd
import scanpy as sc
from scipy import sparse

# Define expression threshold for ebf2
threshold_ebf2 = 1.5

annotation_col = "region_annotation"
new_label = "Early Neuron"

# Extract ebf2 expression from adata_0
ebf2_expression = adata_0[:, "ebf2"].X

if sparse.issparse(ebf2_expression):
    ebf2_expression = ebf2_expression.toarray()

# Cells expressing ebf2 above the threshold
ebf2_mask = (
    np.asarray(ebf2_expression).ravel()
    > threshold_ebf2
)

# Ensure the annotation column is categorical
if not isinstance(
    adata_0.obs[annotation_col].dtype,
    pd.CategoricalDtype
):
    adata_0.obs[annotation_col] = pd.Categorical(
        adata_0.obs[annotation_col]
    )

# Add Early Neuron as an allowed category
if new_label not in adata_0.obs[annotation_col].cat.categories:
    adata_0.obs[annotation_col] = (
        adata_0.obs[annotation_col]
        .cat.add_categories([new_label])
    )

# Assign Early Neuron regardless of the current annotation
adata_0.obs.loc[
    ebf2_mask,
    annotation_col
] = new_label

# Check counts
print("Early Neuron assigned:", ebf2_mask.sum())
print(adata_0.obs[annotation_col].value_counts())

# Visualize
sc.pl.umap(
    adata_0,
    color=annotation_col,
    add_outline=True,
    palette="tab20"
)

In [ ]:
import numpy as np
import pandas as pd
import scanpy as sc
from scipy import sparse

# Define expression threshold for pax2
threshold_pax2 = 1.0

annotation_col = "region_annotation"
new_label = "Hindbrain Primordium"

# Extract pax2 expression from adata_0
pax2_expression = adata_0[:, "pax2"].X

if sparse.issparse(pax2_expression):
    pax2_expression = pax2_expression.toarray()

# Cells expressing pax2 above the threshold
pax2_mask = (
    np.asarray(pax2_expression).ravel()
    > threshold_pax2
)

# Ensure the annotation column is categorical
if not isinstance(
    adata_0.obs[annotation_col].dtype,
    pd.CategoricalDtype
):
    adata_0.obs[annotation_col] = pd.Categorical(
        adata_0.obs[annotation_col]
    )

# Add Neural Plate HB as an allowed category
if new_label not in adata_0.obs[annotation_col].cat.categories:
    adata_0.obs[annotation_col] = (
        adata_0.obs[annotation_col]
        .cat.add_categories([new_label])
    )

# Allow overwriting only Neural Plate or Other
allowed_mask = (
    adata_0.obs[annotation_col]
    .isin([
        "Neural Plate 2",
        "Other"
    ])
    .to_numpy()
)

# High pax2 expression and an eligible current annotation
neural_plate_hb_mask = pax2_mask & allowed_mask

# Assign Neural Plate HB
adata_0.obs.loc[
    neural_plate_hb_mask,
    annotation_col
] = new_label

# Check counts
print(
    "Hindbrain Primordium assigned:",
    neural_plate_hb_mask.sum()
)
print(adata_0.obs[annotation_col].value_counts())

# Visualize
sc.pl.umap(
    adata_0,
    color=annotation_col,
    add_outline=True,
    palette="tab20"
)

In [ ]:
import numpy as np
import pandas as pd
import scanpy as sc
from scipy import sparse

# Define expression threshold for foxa4
threshold_foxa1 = 1.0

annotation_col = "region_annotation"
new_label = "Basal Cells"

# Extract foxa4 expression from adata_0
foxa1_expression = adata_0[:, "foxa1"].X

if sparse.issparse(foxa1_expression):
    foxa1_expression = foxa1_expression.toarray()

# Cells expressing foxa4 above the threshold
foxa1_mask = (
    np.asarray(foxa1_expression).ravel()
    > threshold_foxa1
)

# Ensure the annotation column is categorical
if not isinstance(
    adata_0.obs[annotation_col].dtype,
    pd.CategoricalDtype
):
    adata_0.obs[annotation_col] = pd.Categorical(
        adata_0.obs[annotation_col]
    )

# Add Basal Cells as an allowed category
if new_label not in adata_0.obs[annotation_col].cat.categories:
    adata_0.obs[annotation_col] = (
        adata_0.obs[annotation_col]
        .cat.add_categories([new_label])
    )

# Assign Basal Cells regardless of the current annotation
adata_0.obs.loc[
    foxa1_mask,
    annotation_col
] = new_label

# Check counts
print("Basal Cells assigned:", foxa1_mask.sum())
print(adata_0.obs[annotation_col].value_counts())

# Visualize
sc.pl.umap(
    adata_0,
    color=annotation_col,
    add_outline=True,
    palette="tab20"
)

In [ ]:
import numpy as np
import pandas as pd
import scanpy as sc
from scipy import sparse

# Define expression threshold for rpe65
threshold_rpe65 = 1.0

annotation_col = "region_annotation"
new_label = "Anterior Neural Crest"

# Extract rpe65 expression from adata_0
rpe65_expression = adata_0[:, "rpe65"].X

if sparse.issparse(rpe65_expression):
    rpe65_expression = rpe65_expression.toarray()

# Cells expressing rpe65 above the threshold
rpe65_mask = (
    np.asarray(rpe65_expression).ravel()
    > threshold_rpe65
)

# Ensure region_annotation is categorical
if not isinstance(
    adata_0.obs[annotation_col].dtype,
    pd.CategoricalDtype
):
    adata_0.obs[annotation_col] = pd.Categorical(
        adata_0.obs[annotation_col]
    )

# Add Anterior Neural Crest as a category if needed
if new_label not in adata_0.obs[annotation_col].cat.categories:
    adata_0.obs[annotation_col] = (
        adata_0.obs[annotation_col]
        .cat.add_categories([new_label])
    )

# Restrict assignment to cells currently labeled Neural Crest
neural_crest_mask = (
    adata_0.obs[annotation_col]
    .eq("Neural Crest")
    .to_numpy()
)

# High rpe65 expression and currently Neural Crest
anterior_neural_crest_mask = (
    rpe65_mask
    & neural_crest_mask
)

# Assign Anterior Neural Crest
adata_0.obs.loc[
    anterior_neural_crest_mask,
    annotation_col
] = new_label

# Check counts
print(
    "Anterior Neural Crest assigned:",
    anterior_neural_crest_mask.sum()
)
print(adata_0.obs[annotation_col].value_counts())

# Visualize
sc.pl.umap(
    adata_0,
    color=annotation_col,
    add_outline=True,
    palette="tab20"
)

In [ ]:
import numpy as np
import pandas as pd
import scanpy as sc
from scipy import sparse

# Define expression threshold for mafb
threshold_mafb = 1.25

annotation_col = "region_annotation"
new_label = "Posterior Neural Crest"

# Extract mafb expression from adata_0
mafb_expression = adata_0[:, "mafb"].X

if sparse.issparse(mafb_expression):
    mafb_expression = mafb_expression.toarray()

# Cells expressing mafb above the threshold
mafb_mask = (
    np.asarray(mafb_expression).ravel()
    > threshold_mafb
)

# Ensure region_annotation is categorical
if not isinstance(
    adata_0.obs[annotation_col].dtype,
    pd.CategoricalDtype
):
    adata_0.obs[annotation_col] = pd.Categorical(
        adata_0.obs[annotation_col]
    )

# Add Posterior Neural Crest as a category if needed
if new_label not in adata_0.obs[annotation_col].cat.categories:
    adata_0.obs[annotation_col] = (
        adata_0.obs[annotation_col]
        .cat.add_categories([new_label])
    )

# Restrict assignment to cells currently labeled Neural Crest
neural_crest_mask = (
    adata_0.obs[annotation_col]
    .eq("Neural Crest")
    .to_numpy()
)

# High mafb expression and currently Neural Crest
posterior_neural_crest_mask = (
    mafb_mask
    & neural_crest_mask
)

# Assign Posterior Neural Crest
adata_0.obs.loc[
    posterior_neural_crest_mask,
    annotation_col
] = new_label

# Check counts
print(
    "Posterior Neural Crest assigned:",
    posterior_neural_crest_mask.sum()
)
print(adata_0.obs[annotation_col].value_counts())

# Visualize
sc.pl.umap(
    adata_0,
    color=annotation_col,
    add_outline=True,
    palette="tab20"
)

__Performing an additional Knn calculation on annotation labels to eliminate the "Other" annotation__

In [ ]:
import numpy as np
import pandas as pd
import scanpy as sc
from sklearn.neighbors import NearestNeighbors

annotation_col = "region_annotation"
unassigned_label = "Other"

# Number of nearby cells in the existing UMAP to examine
n_neighbors = 5

# Number of iterative labeling rounds
maximum_iterations = 10

# Required fraction of the weighted vote
minimum_confidence = 0.25

# Preserve the existing UMAP coordinates
original_umap = adata_0.obsm["X_umap"].copy()

# Save labels before this procedure
adata_0.obs["region_annotation_before_umap_knn"] = (
    adata_0.obs[annotation_col].astype(str)
)

labels = adata_0.obs[annotation_col].astype(str).copy()

# Build KNN using the existing UMAP coordinates
knn = NearestNeighbors(
    n_neighbors=n_neighbors + 1,
    metric="euclidean"
)

knn.fit(original_umap)

distances, neighbor_indices = knn.kneighbors(original_umap)

# Remove each cell itself from its neighbor list
distances = distances[:, 1:]
neighbor_indices = neighbor_indices[:, 1:]

for iteration in range(maximum_iterations):

    current_labels = labels.copy()

    other_positions = np.flatnonzero(
        current_labels.eq(unassigned_label).to_numpy()
    )

    assignments = {}

    for cell_pos in other_positions:

        cell_neighbor_positions = neighbor_indices[cell_pos]
        cell_neighbor_distances = distances[cell_pos]

        neighbor_labels = (
            current_labels
            .iloc[cell_neighbor_positions]
            .to_numpy()
        )

        # Ignore neighbors that are also Other
        labeled_mask = neighbor_labels != unassigned_label

        neighbor_labels = neighbor_labels[labeled_mask]
        neighbor_distances = cell_neighbor_distances[labeled_mask]

        if len(neighbor_labels) == 0:
            continue

        # Nearby cells receive greater voting weight
        neighbor_weights = 1 / (neighbor_distances + 1e-8)

        weighted_votes = (
            pd.DataFrame({
                "label": neighbor_labels,
                "weight": neighbor_weights
            })
            .groupby("label")["weight"]
            .sum()
            .sort_values(ascending=False)
        )

        best_label = weighted_votes.index[0]

        confidence = (
            weighted_votes.iloc[0]
            / weighted_votes.sum()
        )

        if confidence >= minimum_confidence:
            assignments[cell_pos] = best_label

    if not assignments:
        print(
            f"Iteration {iteration + 1}: "
            "no additional Other cells could be assigned."
        )
        break

    for cell_pos, new_label in assignments.items():
        labels.iloc[cell_pos] = new_label

    print(
        f"Iteration {iteration + 1}: "
        f"assigned {len(assignments)} cells; "
        f"{labels.eq(unassigned_label).sum()} remain Other."
    )

# Store the updated annotations
adata_0.obs[annotation_col] = pd.Categorical(labels)

# Explicitly preserve the original embedding
adata_0.obsm["X_umap"] = original_umap

print("\nFinal annotation counts:")
print(adata_0.obs[annotation_col].value_counts())

# Plot using the unchanged UMAP coordinates
sc.pl.umap(
    adata_0,
    color=annotation_col,
    frameon=False,
    size=30,
    add_outline=False,
    alpha=0.5,
    save="_S04_08_umap_annotation_col.png"
)

In [ ]:
import numpy as np
import pandas as pd
import scanpy as sc
from sklearn.neighbors import NearestNeighbors

annotation_col = "region_annotation"
unassigned_label = "Other"

# Local boundary-smoothing settings
n_neighbors = 5
minimum_confidence = 0.50
minimum_labeled_neighbors = 5
maximum_iterations = 5

# ------------------------------------------------------------
# Preserve labels and the existing UMAP
# ------------------------------------------------------------

if "X_umap" not in adata_0.obsm:
    raise KeyError("adata_0.obsm['X_umap'] does not exist.")

umap_coordinates = adata_0.obsm["X_umap"].copy()

adata_0.obs["region_annotation_before_smoothing"] = (
    adata_0.obs[annotation_col].astype(str)
)

labels = adata_0.obs[annotation_col].astype(str).copy()

# Track which cells changed
last_confidence = pd.Series(
    np.nan,
    index=adata_0.obs_names,
    dtype=float
)

# ------------------------------------------------------------
# Construct neighbors using the existing UMAP coordinates
# ------------------------------------------------------------

knn = NearestNeighbors(
    n_neighbors=n_neighbors + 1,
    metric="euclidean"
)

knn.fit(umap_coordinates)

distances, neighbor_indices = knn.kneighbors(umap_coordinates)

# Remove each cell itself
distances = distances[:, 1:]
neighbor_indices = neighbor_indices[:, 1:]

# ------------------------------------------------------------
# Iteratively smooth all annotation boundaries
# ------------------------------------------------------------

for iteration in range(maximum_iterations):

    # Snapshot prevents changes from propagating within the same iteration
    current_labels = labels.copy()

    assignments = {}
    confidences = {}

    for cell_pos in range(adata_0.n_obs):

        cell_neighbor_positions = neighbor_indices[cell_pos]
        cell_neighbor_distances = distances[cell_pos]

        neighbor_labels = (
            current_labels
            .iloc[cell_neighbor_positions]
            .to_numpy()
        )

        # Do not allow Other cells to determine the winning annotation
        valid_neighbor_mask = neighbor_labels != unassigned_label

        neighbor_labels = neighbor_labels[valid_neighbor_mask]
        neighbor_distances = cell_neighbor_distances[valid_neighbor_mask]

        if len(neighbor_labels) < minimum_labeled_neighbors:
            continue

        # Nearby neighbors receive more weight
        neighbor_weights = 1 / (neighbor_distances + 1e-8)

        weighted_votes = (
            pd.DataFrame({
                "label": neighbor_labels,
                "weight": neighbor_weights
            })
            .groupby("label", observed=True)["weight"]
            .sum()
            .sort_values(ascending=False)
        )

        best_label = weighted_votes.index[0]

        confidence = (
            weighted_votes.iloc[0]
            / weighted_votes.sum()
        )

        current_label = current_labels.iloc[cell_pos]

        # Change the label only when the local vote is sufficiently strong
        if (
            confidence >= minimum_confidence
            and best_label != current_label
        ):
            assignments[cell_pos] = best_label
            confidences[cell_pos] = confidence

    if not assignments:
        print(
            f"Iteration {iteration + 1}: "
            "labels have stabilized."
        )
        break

    # Apply all changes simultaneously
    for cell_pos, new_label in assignments.items():
        labels.iloc[cell_pos] = new_label

        cell_name = adata_0.obs_names[cell_pos]
        last_confidence.loc[cell_name] = confidences[cell_pos]

    print(
        f"Iteration {iteration + 1}: "
        f"{len(assignments)} cells changed labels."
    )

# ------------------------------------------------------------
# Save smoothed annotations
# ------------------------------------------------------------

adata_0.obs[annotation_col] = pd.Categorical(labels)

adata_0.obs["boundary_smoothing_confidence"] = (
    last_confidence
)

before_labels = (
    adata_0.obs["region_annotation_before_smoothing"]
    .astype(str)
)

after_labels = (
    adata_0.obs[annotation_col]
    .astype(str)
)

adata_0.obs["label_changed_by_smoothing"] = (
    before_labels != after_labels
)

# Ensure the UMAP coordinates remain unchanged
assert np.array_equal(
    adata_0.obsm["X_umap"],
    umap_coordinates
)

# ------------------------------------------------------------
# Display changes
# ------------------------------------------------------------

changed_mask = adata_0.obs["label_changed_by_smoothing"]

print("\nTotal cells relabeled:")
print(changed_mask.sum())

print("\nLabel changes:")
print(
    pd.crosstab(
        before_labels[changed_mask],
        after_labels[changed_mask],
        rownames=["Original label"],
        colnames=["New label"]
    )
)

print("\nFinal annotation counts:")
print(
    adata_0.obs[annotation_col].value_counts()
)

# ------------------------------------------------------------
# Plot on the unchanged UMAP
# ------------------------------------------------------------

sc.pl.umap(
    adata_0,
    color=annotation_col,
    frameon=False,
    size=30,
    add_outline=False,
    alpha=0.5,
    save="_S04_09_umap_annotation_col.png"
)

__Recombine our annotated subsetted data with our original adata UMAP under the column "region_annotation"__

In [ ]:
# Extract unique categories from adata_0
new_categories_3 = adata_0.obs['region_annotation'].unique()

# Get the current categories in adata
current_categories_R = adata.obs['region_annotation'].cat.categories

# Filter out the new categories that already exist in adata
categories_to_add = [cat for cat in new_categories_3 if cat not in current_categories_R]

# If there are new categories to add, update adata.obs['category'] to include these new categories
if categories_to_add:
    adata.obs['region_annotation'] = adata.obs['region_annotation'].cat.add_categories(categories_to_add)

# Now proceed to update the 'category' in adata based on annotations in adata_0
for cell_id in adata_0.obs_names:
    # Ensure the cell_id exists in adata to avoid KeyErrors
    if cell_id in adata.obs_names:
        # Update the category in adata with the category from adata_3
        adata.obs.at[cell_id, 'region_annotation'] = adata_0.obs.at[cell_id, 'region_annotation']

__Alphabetically order our categories__

In [ ]:
# Ensure 'category' is a Categorical datatype
adata.obs['region_annotation'] = adata.obs['region_annotation'].astype('category')

# Sort the categories alphabetically
adata.obs['region_annotation'] = adata.obs['region_annotation'].cat.reorder_categories(sorted(adata.obs['region_annotation'].cat.categories), ordered=True)

In [ ]:
# Now, plot the UMAP
sc.pl.umap(adata, color='region_annotation', frameon=False, size=20, add_outline=False, alpha=.5, save="_S04_10_umap_region_annotation.png")

In [ ]:
sc.pl.umap(adata, color='leiden_0.2', frameon=False, size=20, add_outline=False, alpha=.5, save="_S04_11_umap_leiden_0_2.png")

In [ ]:
adata_4 = adata[
    adata.obs["leiden_0.2"].astype(str).isin(['0', '5', '9'])
].copy()



In [ ]:
#Markers Expression Check
gene_list=['krt7', 'grhl1', 'sox2', 'sox3', 'tbxt', 'fgf8', 'hoxd3', 'myf5', 'six1', 'eya1', 'atp6v1b1', 'actc1', 'tbx1', 'foxc1', 'shh', 'gsc']
sc.pl.umap(adata_4, color=gene_list,  color_map=plt.cm.turbo, add_outline=True, size=20, frameon=False, legend_fontsize=12, alpha=1, save="_S04_12_umap_marker_panel.png")

In [ ]:
sc.pl.umap(adata_4, frameon=False, size=20, add_outline=True, save="_S04_13_umap.png")

In [ ]:
import numpy as np
import scanpy as sc
from scipy import sparse

# Define expression threshold
threshold = 0.05

annotation_col = "region_annotation"
new_label = "Pronephric Primordium"

# Extract pax8 expression
pax8_expression = adata_4[:, "pax8"].X
if sparse.issparse(pax8_expression):
    pax8_expression = pax8_expression.toarray()

pax8_mask = np.asarray(pax8_expression).ravel() > threshold

# Extract lhx1 expression
lhx1_expression = adata_4[:, "lhx1"].X
if sparse.issparse(lhx1_expression):
    lhx1_expression = lhx1_expression.toarray()

lhx1_mask = np.asarray(lhx1_expression).ravel() > threshold

# Require both pax8 and lhx1 expression
pronephric_mask = pax8_mask & lhx1_mask

# Reset all annotations to Other
adata_4.obs[annotation_col] = "Other"

# Assign Pronephric Primordium
adata_4.obs.loc[
    pronephric_mask,
    annotation_col
] = new_label

# Check annotation counts
print("Pronephric Primordium assigned:", pronephric_mask.sum())
print(adata_4.obs[annotation_col].value_counts())

# Visualize
sc.pl.umap(
    adata_4,
    color=annotation_col,
    add_outline=True,
    palette="tab20"
)

In [ ]:
import numpy as np
import pandas as pd
import scanpy as sc
from scipy import sparse

# Define expression threshold for actc1
threshold_actc1 = 0.25

annotation_col = "region_annotation"
new_label = "Anterior Paraxial Mesoderm"

# Extract actc1 expression from adata_4
actc1_expression = adata_4[:, "actc1"].X

if sparse.issparse(actc1_expression):
    actc1_expression = actc1_expression.toarray()

# Cells expressing actc1 above the threshold
actc1_mask = (
    np.asarray(actc1_expression).ravel()
    > threshold_actc1
)

# Ensure the annotation column is categorical
if not isinstance(
    adata_4.obs[annotation_col].dtype,
    pd.CategoricalDtype
):
    adata_4.obs[annotation_col] = pd.Categorical(
        adata_4.obs[annotation_col]
    )

# Add Anterior Paraxial Mesoderm as a category
if new_label not in adata_4.obs[annotation_col].cat.categories:
    adata_4.obs[annotation_col] = (
        adata_4.obs[annotation_col]
        .cat.add_categories([new_label])
    )

# Assign the label regardless of the current annotation
adata_4.obs.loc[
    actc1_mask,
    annotation_col
] = new_label

# Check counts
print(
    "Anterior Paraxial Mesoderm assigned:",
    actc1_mask.sum()
)
print(adata_4.obs[annotation_col].value_counts())

# Visualize
sc.pl.umap(
    adata_4,
    color=annotation_col,
    add_outline=True,
    palette="tab20"
)

In [ ]:
import numpy as np
import pandas as pd
import scanpy as sc
from scipy import sparse

# Define expression thresholds
threshold_myf5 = 1.0
threshold_hoxd3 = 1.0

annotation_col = "region_annotation"
new_label = "Posterior Paraxial Mesoderm"

# Extract myf5 expression from adata_4
myf5_expression = adata_4[:, "myf5"].X

if sparse.issparse(myf5_expression):
    myf5_expression = myf5_expression.toarray()

myf5_mask = (
    np.asarray(myf5_expression).ravel()
    > threshold_myf5
)

# Extract hoxd3 expression from adata_4
hoxd3_expression = adata_4[:, "hoxd3"].X

if sparse.issparse(hoxd3_expression):
    hoxd3_expression = hoxd3_expression.toarray()

hoxd3_mask = (
    np.asarray(hoxd3_expression).ravel()
    > threshold_hoxd3
)

# Require both myf5 and hoxd3 expression
posterior_para_mask = myf5_mask & hoxd3_mask

# Ensure the annotation column is categorical
if not isinstance(
    adata_4.obs[annotation_col].dtype,
    pd.CategoricalDtype
):
    adata_4.obs[annotation_col] = pd.Categorical(
        adata_4.obs[annotation_col]
    )

# Add Posterior Paraxial Mesoderm as a category
if new_label not in adata_4.obs[annotation_col].cat.categories:
    adata_4.obs[annotation_col] = (
        adata_4.obs[annotation_col]
        .cat.add_categories([new_label])
    )

# Assign the label regardless of the current annotation
adata_4.obs.loc[
    posterior_para_mask,
    annotation_col
] = new_label

# Check counts
print(
    "Posterior Paraxial Mesoderm assigned:",
    posterior_para_mask.sum()
)
print(adata_4.obs[annotation_col].value_counts())

# Visualize
sc.pl.umap(
    adata_4,
    color=annotation_col,
    add_outline=True,
    palette="tab20"
)

In [ ]:
import numpy as np
import pandas as pd
import scanpy as sc
from scipy import sparse

# Define expression threshold for tbx1
threshold_tbx1 = 1.0

annotation_col = "region_annotation"
new_label = "Pharyngeal Mesoderm"

# Extract tbx1 expression from adata_4
tbx1_expression = adata_4[:, "tbx1"].X

if sparse.issparse(tbx1_expression):
    tbx1_expression = tbx1_expression.toarray()

# Cells expressing tbx1 above the threshold
tbx1_mask = (
    np.asarray(tbx1_expression).ravel()
    > threshold_tbx1
)

# Ensure the annotation column is categorical
if not isinstance(
    adata_4.obs[annotation_col].dtype,
    pd.CategoricalDtype
):
    adata_4.obs[annotation_col] = pd.Categorical(
        adata_4.obs[annotation_col]
    )

# Add Pharyngeal Mesoderm as a category
if new_label not in adata_4.obs[annotation_col].cat.categories:
    adata_4.obs[annotation_col] = (
        adata_4.obs[annotation_col]
        .cat.add_categories([new_label])
    )

# Assign Pharyngeal Mesoderm regardless of the current annotation
adata_4.obs.loc[
    tbx1_mask,
    annotation_col
] = new_label

# Check counts
print("Pharyngeal Mesoderm assigned:", tbx1_mask.sum())
print(adata_4.obs[annotation_col].value_counts())

# Visualize
sc.pl.umap(
    adata_4,
    color=annotation_col,
    add_outline=True,
    palette="tab20"
)

In [ ]:
import numpy as np
import pandas as pd
import scanpy as sc
from scipy import sparse

# Define expression threshold for runx1
threshold_runx1 = 1.0

annotation_col = "region_annotation"
new_label = "Ventral Blood Island"

# Extract runx1 expression from adata_4
runx1_expression = adata_4[:, "runx1"].X

if sparse.issparse(runx1_expression):
    runx1_expression = runx1_expression.toarray()

# Cells expressing runx1 above the threshold
runx1_mask = (
    np.asarray(runx1_expression).ravel()
    > threshold_runx1
)

# Ensure the annotation column is categorical
if not isinstance(
    adata_4.obs[annotation_col].dtype,
    pd.CategoricalDtype
):
    adata_4.obs[annotation_col] = pd.Categorical(
        adata_4.obs[annotation_col]
    )

# Add Ventral Blood Island as a category
if new_label not in adata_4.obs[annotation_col].cat.categories:
    adata_4.obs[annotation_col] = (
        adata_4.obs[annotation_col]
        .cat.add_categories([new_label])
    )

# Assign Ventral Blood Island regardless of the current annotation
adata_4.obs.loc[
    runx1_mask,
    annotation_col
] = new_label

# Check counts
print("Ventral Blood Island assigned:", runx1_mask.sum())
print(adata_4.obs[annotation_col].value_counts())

# Visualize
sc.pl.umap(
    adata_4,
    color=annotation_col,
    add_outline=True,
    palette="tab20"
)

In [ ]:
import numpy as np
import pandas as pd
import scanpy as sc
from scipy import sparse

# Define expression threshold for gata6
threshold_gata6 = 1.0

annotation_col = "region_annotation"
new_label = "Ventral Mesoderm"

# Extract gata6 expression from adata_4
gata6_expression = adata_4[:, "gata6"].X

if sparse.issparse(gata6_expression):
    gata6_expression = gata6_expression.toarray()

# Cells expressing gata6 above the threshold
gata6_mask = (
    np.asarray(gata6_expression).ravel()
    > threshold_gata6
)

# Ensure the annotation column is categorical
if not isinstance(
    adata_4.obs[annotation_col].dtype,
    pd.CategoricalDtype
):
    adata_4.obs[annotation_col] = pd.Categorical(
        adata_4.obs[annotation_col]
    )

# Add Ventral Mesoderm as a category
if new_label not in adata_4.obs[annotation_col].cat.categories:
    adata_4.obs[annotation_col] = (
        adata_4.obs[annotation_col]
        .cat.add_categories([new_label])
    )

# Protect these existing annotations
protected_labels = [
    "Ventral Blood Island",
    "Pronephric Primordium",
    "Cardiac Mesoderm"
]

not_protected_mask = (
    ~adata_4.obs[annotation_col]
    .isin(protected_labels)
    .to_numpy()
)

# High gata6 expression and not a protected annotation
ventral_mesoderm_mask = (
    gata6_mask
    & not_protected_mask
)

# Assign Ventral Mesoderm
adata_4.obs.loc[
    ventral_mesoderm_mask,
    annotation_col
] = new_label

# Check counts
print(
    "Ventral Mesoderm assigned:",
    ventral_mesoderm_mask.sum()
)
print(
    "gata6-positive protected cells preserved:",
    (gata6_mask & ~not_protected_mask).sum()
)
print(adata_4.obs[annotation_col].value_counts())

# Visualize
sc.pl.umap(
    adata_4,
    color=annotation_col,
    add_outline=True,
    palette="tab20"
)

In [ ]:
import numpy as np
import pandas as pd
import scanpy as sc
from scipy import sparse

# Define expression threshold for not
threshold_not = 1.0

annotation_col = "region_annotation"
new_label = "Notochord"

# Extract not expression from adata_4
not_expression = adata_4[:, "not"].X

if sparse.issparse(not_expression):
    not_expression = not_expression.toarray()

# Cells expressing not above the threshold
not_mask = (
    np.asarray(not_expression).ravel()
    > threshold_not
)

# Ensure the annotation column is categorical
if not isinstance(
    adata_4.obs[annotation_col].dtype,
    pd.CategoricalDtype
):
    adata_4.obs[annotation_col] = pd.Categorical(
        adata_4.obs[annotation_col]
    )

# Add Notochord as a category
if new_label not in adata_4.obs[annotation_col].cat.categories:
    adata_4.obs[annotation_col] = (
        adata_4.obs[annotation_col]
        .cat.add_categories([new_label])
    )

# Assign Notochord regardless of the current annotation
adata_4.obs.loc[
    not_mask,
    annotation_col
] = new_label

# Check counts
print("Notochord assigned:", not_mask.sum())
print(adata_4.obs[annotation_col].value_counts())

# Visualize
sc.pl.umap(
    adata_4,
    color=annotation_col,
    add_outline=True,
    palette="tab20"
)

In [ ]:
import numpy as np
import pandas as pd
import scanpy as sc
from scipy import sparse

# Define expression thresholds
threshold_foxc1 = 0.1
threshold_tbxt = 1.0

annotation_col = "region_annotation"
new_label = "Lateral Mesoderm"

# Extract foxc1 expression from adata_4
foxc1_expression = adata_4[:, "foxc1"].X

if sparse.issparse(foxc1_expression):
    foxc1_expression = foxc1_expression.toarray()

foxc1_mask = (
    np.asarray(foxc1_expression).ravel()
    > threshold_foxc1
)

# Extract tbxt expression from adata_4
tbxt_expression = adata_4[:, "tbxt"].X

if sparse.issparse(tbxt_expression):
    tbxt_expression = tbxt_expression.toarray()

tbxt_mask = (
    np.asarray(tbxt_expression).ravel()
    > threshold_tbxt
)

# Require foxc1 OR tbxt expression
combined_mask = foxc1_mask | tbxt_mask

# Ensure the annotation column is categorical
if not isinstance(
    adata_4.obs[annotation_col].dtype,
    pd.CategoricalDtype
):
    adata_4.obs[annotation_col] = pd.Categorical(
        adata_4.obs[annotation_col]
    )

# Add Lateral Mesoderm as a category if needed
if new_label not in adata_4.obs[annotation_col].cat.categories:
    adata_4.obs[annotation_col] = (
        adata_4.obs[annotation_col]
        .cat.add_categories([new_label])
    )

# Only overwrite cells currently labeled Other
other_mask = (
    adata_4.obs[annotation_col]
    .eq("Other")
    .to_numpy()
)

lateral_mesoderm_mask = combined_mask & other_mask

# Assign Lateral Mesoderm
adata_4.obs.loc[
    lateral_mesoderm_mask,
    annotation_col
] = new_label

# Check counts
print("Lateral Mesoderm assigned:", lateral_mesoderm_mask.sum())
print(adata_4.obs[annotation_col].value_counts())

# Visualize
sc.pl.umap(
    adata_4,
    color=annotation_col,
    add_outline=True,
    palette="tab20"
)

In [ ]:
import numpy as np
import pandas as pd
import scanpy as sc
from scipy import sparse

# Define expression threshold for gsc
threshold_gsc = 0.1  # Adjust as needed

annotation_col = "region_annotation"
new_label = "Pre-chordal Plate Mesoderm"

# Extract gsc expression from adata_4
gsc_expression = adata_4[:, "gsc"].X

if sparse.issparse(gsc_expression):
    gsc_expression = gsc_expression.toarray()

# Cells expressing gsc above the threshold
gsc_mask = (
    np.asarray(gsc_expression).ravel()
    > threshold_gsc
)

# Ensure the annotation column is categorical
if not isinstance(
    adata_4.obs[annotation_col].dtype,
    pd.CategoricalDtype
):
    adata_4.obs[annotation_col] = pd.Categorical(
        adata_4.obs[annotation_col]
    )

# Add Lateral Mesoderm as a category if needed
if new_label not in adata_4.obs[annotation_col].cat.categories:
    adata_4.obs[annotation_col] = (
        adata_4.obs[annotation_col]
        .cat.add_categories([new_label])
    )

# Assign Lateral Mesoderm regardless of the current annotation
adata_4.obs.loc[
    gsc_mask,
    annotation_col
] = new_label

# Check counts
print("Pre-chordal Plate Mesoderm assigned:", gsc_mask.sum())
print(adata_4.obs[annotation_col].value_counts())

# Visualize
sc.pl.umap(
    adata_4,
    color=annotation_col,
    add_outline=True,
    palette="tab20"
)

__Performing an additional Knn calculation on annotation labels to eliminate the "Other" annotation__

In [ ]:
import numpy as np
import pandas as pd
import scanpy as sc
from sklearn.neighbors import NearestNeighbors

annotation_col = "region_annotation"
unassigned_label = "Other"

# Number of nearby cells in the existing UMAP to examine
n_neighbors = 5

# Number of iterative labeling rounds
maximum_iterations = 10

# Required fraction of the weighted vote
minimum_confidence = 0.25

# Preserve the existing UMAP coordinates
original_umap = adata_4.obsm["X_umap"].copy()

# Save labels before this procedure
adata_4.obs["region_annotation_before_umap_knn"] = (
    adata_4.obs[annotation_col].astype(str)
)

labels = adata_4.obs[annotation_col].astype(str).copy()

# Build KNN using the existing UMAP coordinates
knn = NearestNeighbors(
    n_neighbors=n_neighbors + 1,
    metric="euclidean"
)

knn.fit(original_umap)

distances, neighbor_indices = knn.kneighbors(original_umap)

# Remove each cell itself from its neighbor list
distances = distances[:, 1:]
neighbor_indices = neighbor_indices[:, 1:]

for iteration in range(maximum_iterations):

    current_labels = labels.copy()

    other_positions = np.flatnonzero(
        current_labels.eq(unassigned_label).to_numpy()
    )

    assignments = {}

    for cell_pos in other_positions:

        cell_neighbor_positions = neighbor_indices[cell_pos]
        cell_neighbor_distances = distances[cell_pos]

        neighbor_labels = (
            current_labels
            .iloc[cell_neighbor_positions]
            .to_numpy()
        )

        # Ignore neighbors that are also Other
        labeled_mask = neighbor_labels != unassigned_label

        neighbor_labels = neighbor_labels[labeled_mask]
        neighbor_distances = cell_neighbor_distances[labeled_mask]

        if len(neighbor_labels) == 0:
            continue

        # Nearby cells receive greater voting weight
        neighbor_weights = 1 / (neighbor_distances + 1e-8)

        weighted_votes = (
            pd.DataFrame({
                "label": neighbor_labels,
                "weight": neighbor_weights
            })
            .groupby("label")["weight"]
            .sum()
            .sort_values(ascending=False)
        )

        best_label = weighted_votes.index[0]

        confidence = (
            weighted_votes.iloc[0]
            / weighted_votes.sum()
        )

        if confidence >= minimum_confidence:
            assignments[cell_pos] = best_label

    if not assignments:
        print(
            f"Iteration {iteration + 1}: "
            "no additional Other cells could be assigned."
        )
        break

    for cell_pos, assigned_label in assignments.items():
        labels.iloc[cell_pos] = assigned_label

    print(
        f"Iteration {iteration + 1}: "
        f"assigned {len(assignments)} cells; "
        f"{labels.eq(unassigned_label).sum()} remain Other."
    )

# Store the updated annotations
adata_4.obs[annotation_col] = pd.Categorical(labels)

# Explicitly preserve the original embedding
adata_4.obsm["X_umap"] = original_umap

print("\nFinal annotation counts:")
print(adata_4.obs[annotation_col].value_counts())

# Plot using the unchanged UMAP coordinates
sc.pl.umap(
    adata_4,
    color=annotation_col,
    frameon=False,
    size=30,
    add_outline=False,
    alpha=0.5,
    save="_S04_14_umap_annotation_col.png"
)

__Recombine our annotated subsetted data with our original adata UMAP under the column "region_annotation"__

In [ ]:
# Extract unique categories from adata_4
new_categories_4 = adata_4.obs["region_annotation"].unique()

# Get the current categories in adata
current_categories_R = (
    adata.obs["region_annotation"].cat.categories
)

# Identify categories present in adata_4 but missing from adata
categories_to_add = [
    category
    for category in new_categories_4
    if category not in current_categories_R
]

# Add missing categories to adata
if categories_to_add:
    adata.obs["region_annotation"] = (
        adata.obs["region_annotation"]
        .cat.add_categories(categories_to_add)
    )

# Update region_annotation in adata using matching cells from adata_4
for cell_id in adata_4.obs_names:

    if cell_id in adata.obs_names:
        adata.obs.at[
            cell_id,
            "region_annotation"
        ] = adata_4.obs.at[
            cell_id,
            "region_annotation"
        ]

print(adata.obs["region_annotation"].value_counts())

In [ ]:
# Now, plot the UMAP
sc.pl.umap(adata, color='region_annotation', frameon=False, size=20, add_outline=False, alpha=.5, save="_S04_15_umap_region_annotation.png", palette="tab20")

In [ ]:
sc.pl.umap(adata, color='leiden_0.1', frameon=False, size=20, add_outline=False, alpha=.5, save="_S04_16_umap_leiden_0_1.png")

In [ ]:
cell_counts = adata.obs["region_annotation"].value_counts()

print(cell_counts)

In [ ]:
# Now, plot the UMAP
sc.pl.umap(adata, color='region_annotation', frameon=False, size=20, add_outline=False, alpha=.5, save="_S04_17_umap_region_annotation.png", palette = "tab20")

# Section 5: Differential gene-expression analysis

This section uses the Wilcoxon rank-sum test to identify:

1. marker genes for every sufficiently populated final annotation versus the remaining cells; and
2. marker genes for selected two-group comparisons.

Each test saves the complete Scanpy result, an FDR/log-fold-change filtered
table, the top-ranked genes per group, group sizes, test parameters, and
descriptive figures.

In [ ]:
from scripts.output_utils import (
    export_rank_genes_groups,
    save_annotation_summary,
    save_current_figure,
    save_dataframe,
    slugify,
)

SECTION_FIGURES_DIR = SECTION_FIGURE_DIRS["Section5"]
SECTION_FIGURES_DIR.mkdir(parents=True, exist_ok=True)
DEG_DIR.mkdir(parents=True, exist_ok=True)
TABLES_DIR.mkdir(parents=True, exist_ok=True)
sc.settings.figdir = str(SECTION_FIGURES_DIR)

ANNOTATION_COL = "region_annotation"
MIN_CELLS_PER_GROUP = 7
TOP_GENES_TO_PLOT = 15

print("Current section figure directory:", SECTION_FIGURES_DIR)
print("Differential-expression table directory:", DEG_DIR)

In [ ]:
# Save final annotation abundance before differential-expression testing.
annotation_counts_path = save_annotation_summary(
    adata,
    output_dir=TABLES_DIR,
    filename=f"{DATASET_SLUG}_final_annotation_counts.csv",
    annotation_col=ANNOTATION_COL,
)

group_counts = (
    adata.obs[ANNOTATION_COL]
    .astype(str)
    .value_counts()
    .rename_axis(ANNOTATION_COL)
    .reset_index(name="cell_count")
)

valid_groups = group_counts.loc[
    group_counts["cell_count"] >= MIN_CELLS_PER_GROUP,
    ANNOTATION_COL,
].tolist()

excluded_groups = group_counts.loc[
    group_counts["cell_count"] < MIN_CELLS_PER_GROUP
].copy()

save_dataframe(
    excluded_groups,
    TABLES_DIR,
    f"{DATASET_SLUG}_deg_groups_excluded_below_{MIN_CELLS_PER_GROUP}_cells.csv",
)

adata_deg = adata[
    adata.obs[ANNOTATION_COL].astype(str).isin(valid_groups)
].copy()

if isinstance(adata_deg.obs[ANNOTATION_COL].dtype, pd.CategoricalDtype):
    adata_deg.obs[ANNOTATION_COL] = (
        adata_deg.obs[ANNOTATION_COL].cat.remove_unused_categories()
    )

all_groups_key = "all_annotations_vs_rest_wilcoxon"

sc.tl.rank_genes_groups(
    adata_deg,
    groupby=ANNOTATION_COL,
    groups=valid_groups,
    reference="rest",
    method="wilcoxon",
    pts=True,
    key_added=all_groups_key,
)

export_rank_genes_groups(
    adata_deg,
    key=all_groups_key,
    output_dir=DEG_DIR,
    prefix=f"{DATASET_SLUG}_all_annotations_vs_rest_wilcoxon",
    groupby=ANNOTATION_COL,
    fdr_max=0.05,
    abs_log2fc_min=1.0,
    top_n=25,
)

sc.pl.rank_genes_groups(
    adata_deg,
    groups=valid_groups,
    n_genes=25,
    sharey=False,
    key=all_groups_key,
    show=False,
)
save_current_figure(
    SECTION_FIGURES_DIR,
    f"S05_01_{DATASET_SLUG}_all_annotations_vs_rest_ranked_genes.png",
)

In [ ]:
# Selected pairwise comparisons from the annotation workflow.
PAIRWISE_COMPARISONS = [('Epidermis 1', 'Epidermis 2', 'epidermis_1_vs_2'), ('Neural Plate 1', 'Neural Plate 2', 'neural_plate_1_vs_2'), ('Neural Plate Border 1', 'Neural Plate Border 2', 'neural_plate_border_1_vs_2')]

comparison_inventory = []

for comparison_index, (group_1, group_2, comparison_slug) in enumerate(
    PAIRWISE_COMPARISONS,
    start=1,
):
    present_labels = set(adata.obs[ANNOTATION_COL].astype(str))

    if group_1 not in present_labels or group_2 not in present_labels:
        comparison_inventory.append(
            {
                "comparison": comparison_slug,
                "group_1": group_1,
                "group_2": group_2,
                "status": "skipped_missing_group",
                "group_1_cells": int(
                    (adata.obs[ANNOTATION_COL].astype(str) == group_1).sum()
                ),
                "group_2_cells": int(
                    (adata.obs[ANNOTATION_COL].astype(str) == group_2).sum()
                ),
            }
        )
        print(
            f"Skipping {comparison_slug}: one or both groups are absent."
        )
        continue

    pair = adata[
        adata.obs[ANNOTATION_COL].astype(str).isin([group_1, group_2])
    ].copy()

    if isinstance(pair.obs[ANNOTATION_COL].dtype, pd.CategoricalDtype):
        pair.obs[ANNOTATION_COL] = (
            pair.obs[ANNOTATION_COL].cat.remove_unused_categories()
        )

    pair_counts = pair.obs[ANNOTATION_COL].astype(str).value_counts()

    if (pair_counts < 2).any():
        comparison_inventory.append(
            {
                "comparison": comparison_slug,
                "group_1": group_1,
                "group_2": group_2,
                "status": "skipped_group_below_2_cells",
                "group_1_cells": int(pair_counts.get(group_1, 0)),
                "group_2_cells": int(pair_counts.get(group_2, 0)),
            }
        )
        continue

    deg_key = f"{comparison_slug}_wilcoxon"

    # With only two groups in the subset, each group versus rest is the
    # reciprocal pairwise comparison.
    sc.tl.rank_genes_groups(
        pair,
        groupby=ANNOTATION_COL,
        groups=[group_1, group_2],
        reference="rest",
        method="wilcoxon",
        pts=True,
        key_added=deg_key,
    )

    export_rank_genes_groups(
        pair,
        key=deg_key,
        output_dir=DEG_DIR,
        prefix=f"{DATASET_SLUG}_{comparison_slug}_wilcoxon",
        groupby=ANNOTATION_COL,
        fdr_max=0.05,
        abs_log2fc_min=1.0,
        top_n=25,
    )

    figure_number = comparison_index + 1

    sc.pl.umap(
        pair,
        color=ANNOTATION_COL,
        frameon=False,
        size=20,
        add_outline=False,
        alpha=0.5,
        show=False,
    )
    save_current_figure(
        SECTION_FIGURES_DIR,
        f"S05_{figure_number:02d}_{DATASET_SLUG}_"
        f"{comparison_slug}_umap.png",
    )

    sc.pl.rank_genes_groups(
        pair,
        groups=[group_1, group_2],
        n_genes=25,
        sharey=False,
        key=deg_key,
        show=False,
    )
    save_current_figure(
        SECTION_FIGURES_DIR,
        f"S05_{figure_number:02d}_{DATASET_SLUG}_"
        f"{comparison_slug}_ranked_genes.png",
    )

    sc.pl.rank_genes_groups_heatmap(
        pair,
        groups=[group_1, group_2],
        n_genes=TOP_GENES_TO_PLOT,
        key=deg_key,
        groupby=ANNOTATION_COL,
        show_gene_labels=True,
        dendrogram=False,
        show=False,
    )
    save_current_figure(
        SECTION_FIGURES_DIR,
        f"S05_{figure_number:02d}_{DATASET_SLUG}_"
        f"{comparison_slug}_heatmap.png",
    )

    sc.pl.rank_genes_groups_dotplot(
        pair,
        groups=[group_1, group_2],
        n_genes=TOP_GENES_TO_PLOT,
        key=deg_key,
        groupby=ANNOTATION_COL,
        dendrogram=False,
        show=False,
    )
    save_current_figure(
        SECTION_FIGURES_DIR,
        f"S05_{figure_number:02d}_{DATASET_SLUG}_"
        f"{comparison_slug}_dotplot.png",
    )

    sc.pl.rank_genes_groups_matrixplot(
        pair,
        groups=[group_1, group_2],
        n_genes=TOP_GENES_TO_PLOT,
        key=deg_key,
        groupby=ANNOTATION_COL,
        dendrogram=False,
        show=False,
    )
    save_current_figure(
        SECTION_FIGURES_DIR,
        f"S05_{figure_number:02d}_{DATASET_SLUG}_"
        f"{comparison_slug}_matrixplot.png",
    )

    comparison_inventory.append(
        {
            "comparison": comparison_slug,
            "group_1": group_1,
            "group_2": group_2,
            "status": "completed",
            "group_1_cells": int(pair_counts.get(group_1, 0)),
            "group_2_cells": int(pair_counts.get(group_2, 0)),
        }
    )

comparison_inventory = pd.DataFrame(comparison_inventory)

save_dataframe(
    comparison_inventory,
    TABLES_DIR,
    f"{DATASET_SLUG}_section5_comparison_inventory.csv",
)

comparison_inventory

** **

# __Section 6: Final cell-type annotation rationale considering differential expression and literature precedence__

 __Topics__
- Discussion for utilizing DEG alongside contemporary literature to further annotate germ layer cell-types

  __DEG of interest__
- Basal/Lamina markers like: col18a1, fn1, pdgfra are more highly expressed in Endoderm2, Neuroectoderm2, Epidermis2, Mesoderm2 than (1).
- Epithelial markers like: epas1, nectin2, epcam in Neuroectoderm1, Mesoderm1 as well as grainyheads/keratins in the Epidermis1 and slc5a8.1, plekhg5 in the Endoderm1 are highly expressed compared to (2).
- Taken together (1) cells are likely the 'Outer' epithelial layer while (2) cells are their 'Inner' basal layer counterparts.


  __Literature Precedence__
- Briggs et al., 2018 (https://www.science.org/doi/10.1126/science.aar5780) landmark paper demonstrated Xenopus tropicalis development through gastrulation. At stage 10 they have identified a single neuroectodermal and a single epidermal cluster.
- Lee et al., 2023 (https://www.science.org/doi/10.1126/sciadv.add5745) mucociliary epithelial development paper demonstrated Xenopus tropicalis ectoderm dissected cells through gastrulation. At stage 10 they identify a single cluster of ectoderm cells.
- Numerous studies by Chalmers concerning early Xenopus tropicalis ectoderm development demonstrate that apical-basal polarity of ectodermal cells forming inner and outer layer cells is established as early as the 64-cell stage and later demarked by differential expression of keratin and grainyhead in the early gastrula (Chalmers et al., 2002 - https://doi.org/10.1016/S1534-5807(02)00113-2, Chalmers et al., 2003 - https://doi.org/10.1242/dev.00490, Chalmers et al., 2005 - https://doi.org/10.1242/dev.01645, Chalmers et al., 2006 - https://doi.org/10.1016/j.mod.2006.04.006).

  __Conclusion__
- Previous single cell sequencing experiments of Xenopus tropicalis early gastrula have not yielded sufficient resolution to identify inner and outer epidermal and neuroectodermal cell states, however there is a compilation of evidence supporting the establishment and differential expression across the inner and outer layer ectoderm.
- Our data specifically confirms Chalmers et al., 2006 in identifying heterogenous "Inner" and "Outer" layer ectoderm cell states while also furthering our understanding of the complete repetoire of genes differentially expressed within these emergent cell states of Xenopus tropicalis early gastrula.


** **
 

In [ ]:
# Route all Scanpy and Matplotlib figures in Section 6 here.
SECTION_FIGURES_DIR = SECTION_FIGURE_DIRS["Section6"]
SECTION_FIGURES_DIR.mkdir(parents=True, exist_ok=True)
sc.settings.figdir = str(SECTION_FIGURES_DIR)

print("Current section figure directory:", SECTION_FIGURES_DIR)


In [ ]:
import pandas as pd
import scanpy as sc

# Create a renamed annotation column while preserving the original category column
adata.obs["region_annotation"] = adata.obs["region_annotation"].copy()

# Rename category labels
rename_dict = {
    "Anterior Paraxial Mesoderm": "Anterior Paraxial Mesoderm",
    "Cement Gland Primordium": "Cement Gland Primordium",
    "Ciliated Epidermal Progenitor":"Ciliated Epidermal Progenitor",
    "Dorsolateral Mesoderm": "Dorsolateral Mesoderm",
    "Notoplate":"Notoplate",
    "Basal Cells":"Basal Cells",

    "Epidermis 1": "Outer Epidermis",
    "Epidermis 2": "Inner Epidermis",
    
    "Eye Primordium": "Eye Primordium",

    "Foregut Primordium": "Foregut Primordium",
    "Goblet Cells": "Goblet Cells",

    "Hindbrain Primordium": "Hindbrain Primordium",
    "Hindgut Primordium": "Hindgut Primordium",
    "Ionocyte": "Ionocyte",
    "L/R Organizer Primordium" : "L/R Organizer Primordium",
    "Lateral Mesoderm":"Lateral Mesoderm",
    "Midgut Primordium": "Midgut Primordium",

    "Neural Plate 1": "Outer Neural Plate",
    "Neural Plate 2": "Inner Neural Plate",

    "Neural Plate Border 1": "Outer Neural Plate Border",
    "Neural Plate Border 2": "Inner Neural Plate Border",

    "Notochord": "Notochord",
    "Pharyngeal Mesoderm":"Pharyngeal Mesoderm",
    "Posterior Paraxial Mesoderm":"Posterior Paraxial Mesoderm",
    "Pre-chordal Plate Mesoderm":"Pre-chordal Plate Mesoderm",
    "Pronephric Primordium":"Pronephric Primordium",
    "Ventral Blood Island":"Ventral Blood Island",
    "Ventral Mesoderm":"Ventral Mesoderm",
    "Pre-placodal Region":"Pre-placodal Region"
    

}

adata.obs["region_annotation"] = (
    adata.obs["region_annotation"]
    .replace(rename_dict)
)

groups_to_plot = [
 "Anterior Paraxial Mesoderm",
    "Cement Gland Primordium",
    "Ciliated Epidermal Progenitor",
    "Dorsolateral Mesoderm",
    "Liver Primordium",
    "Early Neuron",
    "Anterior Pharyngeal Primordium",
    "Anterior Neural Crest",
    "Anterior Pre-placodal Region",
    "Posterior Pre-placodal Region",
    "Basal Cells",
    "Pancreatic Primordium",
    "Posterior Neural Crest",
    "Neural Crest",
    "Outer Epidermis",
    "Inner Epidermis",
    "Anterior Neural Ridge",
    "Eye Primordium",
    "Notoplate",
    "Foregut Primordium",
    "Goblet Cell",
    
    "Hindbrain Primordium",
    "Hindgut Primordium",
    "Ionocyte",
    "L/R Organizer Primordium",
    "Lateral Mesoderm",
    "Midgut Primordium",

    "Outer Neural Plate",
    "Inner Neural Plate",

    "Outer Neural Plate Border",
    "Inner Neural Plate Border",
    "Pre-chordal Plate Mesoderm",
    "Notochord",
    "Pharyngeal Mesoderm",
    "Pre-placodal Region",
    "Posterior Paraxial Mesoderm",
    "Pronephric Primordium",
    "Ventral Blood Island",
    "Ventral Mesoderm",
]

adata_germ_layers = adata[
    adata.obs["region_annotation"].isin(groups_to_plot)
].copy()

adata_germ_layers.obs["region_annotation"] = pd.Categorical(
    adata_germ_layers.obs["region_annotation"],
    categories=groups_to_plot,
    ordered=True
)

sc.pl.umap(
    adata_germ_layers,
    color="region_annotation",
    legend_loc="right margin",
    frameon=False,
    title="Germ Layer Subdomains"
, save="_S06_01_umap_region_annotation.png")

__Save final processed data outputs__


In [ ]:
from scripts.output_utils import (
    save_annotation_summary,
    save_cluster_annotation_crosstab,
    save_label_transition_table,
)

# Final cell-type abundance table.
save_annotation_summary(
    adata,
    output_dir=TABLES_DIR,
    filename=f"{DATASET_SLUG}_final_annotation_counts.csv",
    annotation_col="region_annotation",
)

# Save count and row-percentage cross-tabs for each Leiden result retained
# in the final AnnData object.
leiden_columns = [
    column for column in adata.obs.columns
    if str(column).startswith("leiden")
]

for leiden_column in leiden_columns:
    save_cluster_annotation_crosstab(
        adata,
        cluster_col=leiden_column,
        annotation_col="region_annotation",
        output_dir=TABLES_DIR,
        filename=(
            f"{DATASET_SLUG}_{leiden_column}_by_region_annotation_counts.csv"
        ),
    )
    save_cluster_annotation_crosstab(
        adata,
        cluster_col=leiden_column,
        annotation_col="region_annotation",
        output_dir=TABLES_DIR,
        filename=(
            f"{DATASET_SLUG}_{leiden_column}_by_region_annotation_row_percent.csv"
        ),
        normalize=True,
    )

# Save before/after label transition tables when refinement backups exist.
label_backup_columns = [
    column for column in adata.obs.columns
    if str(column).startswith("region_annotation_before")
]

for before_column in label_backup_columns:
    save_label_transition_table(
        adata,
        before_col=before_column,
        after_col="region_annotation",
        output_dir=TABLES_DIR,
        filename=(
            f"{DATASET_SLUG}_{before_column}_to_region_annotation.csv"
        ),
    )

In [ ]:
# Save final annotated data and complete cell metadata.
OUTPUT_H5AD = PROCESSED_DATA_DIR / f"{DATASET_SLUG}_processed_annotated.h5ad"
OUTPUT_OBS_CSV = PROCESSED_DATA_DIR / f"{DATASET_SLUG}_cell_metadata.csv"

adata.write_h5ad(OUTPUT_H5AD)
adata.obs.to_csv(OUTPUT_OBS_CSV)

print("Saved AnnData:", OUTPUT_H5AD)
print("Saved cell metadata:", OUTPUT_OBS_CSV)